# LeetCode 358 — Rearrange String k Distance Apart (Hard) bằng Python

> **Lưu ý: đây là bài Premium** (bị khóa trên LeetCode, chỉ tài khoản trả phí mới mở và nộp được). Notebook tự chứa đầy đủ đề, ví dụ, constraints và bộ test, nên bạn vẫn học và chạy thử trọn vẹn mà không cần tài khoản Premium.

Skeleton trên LeetCode (Python3):

```python
class Solution:
    def rearrangeString(self, s: str, k: int) -> str:

```

Đề bài rất đời thường: *"xếp lại các chữ cái sao cho hai chữ giống nhau đứng cách nhau ít nhất `k` vị trí"*. Ai cũng thử xếp tay được với chuỗi ngắn như `"aabbcc"`. Cái khó của bài Hard nằm ở chỗ:

1. Có **rất nhiều** đáp án đúng (LeetCode chấp nhận bất kỳ đáp án hợp lệ nào), nên ta phải tự viết **hàm kiểm tra hợp lệ** thay vì so chuỗi.
2. Một chiến lược tham lam "ngây thơ" (cứ chọn chữ nào đang được phép) **sai** ở những chuỗi rất ngắn. Ta phải tìm ra quy tắc chọn đúng: **ưu tiên chữ còn nhiều nhất**.
3. Muốn làm nhanh, ta cần hai cấu trúc phối hợp: **max-heap** (lấy chữ còn nhiều nhất) và **hàng đợi chờ** (cooldown queue — "phòng chờ" giữ chữ vừa dùng đúng `k` lượt).

Notebook đi **từng bậc thang**. Ở mỗi bậc ta trả lời hai câu: *cách cũ vướng ở đâu?* và *mấu chốt nào giúp vượt qua?*

---

## Sau notebook này, bạn sẽ nắm được

1. Hiểu chính xác "khoảng cách ít nhất `k`" bằng chỉ số; dùng `collections.Counter` để đếm chữ và từ điển `last_pos` để nhớ vị trí xuất hiện gần nhất; tự viết **hàm kiểm tra hợp lệ** `is_valid_arrangement`.
2. **Cách 1 — Brute force**: thử mọi hoán vị bằng `itertools.permutations` (chỉ dùng cho chuỗi ngắn) — đây là **trọng tài** để biết một input có khả thi hay không.
3. **Cách 2 — Tham lam quét 26 chữ**: tại mỗi vị trí chọn chữ **còn nhiều nhất** trong số chữ **đang được phép** (đã "nghỉ" đủ). Học mẹo `ord`/`chr`, mảng đếm 26 phần tử, mảng `next_valid`. Độ phức tạp `O(26·n)`. Trước đó ta xem bản tham lam "ngây thơ" sai ở đâu.
4. **Cách 3 (cách chính) — Max-heap + hàng đợi chờ kích thước `k`**: học `heapq` và `deque` từ đầu, trace kỹ `"aabbcc"`, `k = 3`; độ phức tạp `O(n log 26)`.
5. **Cách 4 — Xử lý nhanh `k = 0` / `k = 1`** và vì sao mọi chuỗi đều hợp lệ khi đó.
6. **Bonus**: công thức khả thi `(maxf - 1)·k + cnt_max <= n` — biết ngay có xếp được hay không mà không cần xếp.
7. Code nộp LeetCode, kiểm thử đối chiếu chéo với brute force, lỗi hay gặp, mẹo nhận dạng pattern, bài liên quan (#767, #621, #1054), 6 bài tập và đáp án ở cuối.

Ký hiệu dùng xuyên suốt: `n = len(s)`, `σ` (sigma) = số chữ cái **khác nhau** trong `s` (`σ <= 26`).

## 1. Đề bài (theo LeetCode)

Given a string `s` and an integer `k`, rearrange `s` such that the same characters are **at least** distance `k` from each other. If it is not possible to rearrange the string, return an empty string `""`.

Dịch: cho chuỗi `s` và số nguyên `k`. Hãy **sắp xếp lại** các ký tự của `s` sao cho hai ký tự **giống nhau** bất kỳ cách nhau **ít nhất** `k`. Nếu không thể, trả về chuỗi rỗng `""`.

**Example 1**

```text
Input:  s = "aabbcc", k = 3
Output: "abcabc"
Explanation: The same letters are at least a distance of 3 from each other.
```

**Example 2**

```text
Input:  s = "aaabc", k = 3
Output: ""
Explanation: It is not possible to rearrange the string.
```

**Example 3**

```text
Input:  s = "aaadbbcc", k = 2
Output: "abacabcd"
Explanation: The same letters are at least a distance of 2 from each other.
```

**Constraints**

- `1 <= s.length <= 3 * 10^5`
- `s` consists of only lowercase English letters.
- `0 <= k <= s.length`

Ghi chú quan trọng: khi có nhiều cách xếp hợp lệ, LeetCode chấp nhận **bất kỳ** cách nào. Ví dụ với Example 1, `"acbacb"` hay `"bcabca"` cũng đúng. Output `"abcabc"` chỉ là **một** đáp án mẫu.

## 2. "Khoảng cách ít nhất `k`" nghĩa là gì?

Đánh số vị trí từ `0`. Nếu cùng một chữ xuất hiện ở vị trí `i` và `j` (với `i < j`), **khoảng cách** của chúng là `j - i`. Yêu cầu: `j - i >= k` với **mọi** cặp vị trí của cùng một chữ.

Nhìn output của Example 1, `"abcabc"`, `k = 3`:

| vị trí | 0 | 1 | 2 | 3 | 4 | 5 |
|---|---|---|---|---|---|---|
| chữ | a | b | c | a | b | c |

- `a` ở vị trí `0` và `3` → khoảng cách `3 - 0 = 3 >= 3` ✔
- `b` ở `1` và `4` → `3` ✔; `c` ở `2` và `5` → `3` ✔

Một cách nghĩ dễ nhớ: khoảng cách `k` nghĩa là **giữa** hai chữ giống nhau phải có **ít nhất `k - 1` chữ khác**. Với `k = 3`, giữa hai chữ `a` phải chen được 2 chữ khác (`b`, `c`).

Chỉ cần kiểm tra **hai lần xuất hiện liên tiếp** của mỗi chữ: nếu mỗi cặp liên tiếp đã cách nhau `>= k` thì mọi cặp xa hơn còn cách nhau xa hơn nữa. Cell dưới in khoảng cách giữa các lần xuất hiện liên tiếp.

In [1]:
t = "abcabc"
k = 3
for ch in "abc":
    positions = [i for i, c in enumerate(t) if c == ch]
    gaps = [positions[j + 1] - positions[j] for j in range(len(positions) - 1)]
    print(f"chữ {ch!r}: vị trí {positions}, khoảng cách liên tiếp {gaps}, đủ >= {k}? {all(g >= k for g in gaps)}")

print()
t2 = "abacabcd"        # output Example 3, k = 2
for ch in "abcd":
    positions = [i for i, c in enumerate(t2) if c == ch]
    print(f"chữ {ch!r}: vị trí {positions}")

chữ 'a': vị trí [0, 3], khoảng cách liên tiếp [3], đủ >= 3? True
chữ 'b': vị trí [1, 4], khoảng cách liên tiếp [3], đủ >= 3? True
chữ 'c': vị trí [2, 5], khoảng cách liên tiếp [3], đủ >= 3? True

chữ 'a': vị trí [0, 2, 4]
chữ 'b': vị trí [1, 5]
chữ 'c': vị trí [3, 6]
chữ 'd': vị trí [7]


### Vì sao Example 2 không xếp được?

`s = "aaabc"`, `k = 3`. Chữ `a` xuất hiện 3 lần, nên 3 chữ `a` phải ở các vị trí `p1 < p2 < p3` với `p2 >= p1 + 3` và `p3 >= p2 + 3`. Đặt `a` sớm nhất có thể: `p1 = 0`, `p2 = 3`, `p3 = 6`. Nhưng chuỗi chỉ dài 5 (vị trí `0..4`) — **không đủ chỗ**. Nói cách khác: giữa 3 chữ `a` cần chen `2 + 2 = 4` chữ khác, mà ta chỉ có 2 chữ (`b`, `c`).

Nhận xét này (chữ xuất hiện nhiều nhất là "nút cổ chai") sẽ quay lại ở mục 5 và mục 11.

### Hai trường hợp đặc biệt: `k = 0` và `k = 1`

- `k = 0`: yêu cầu `j - i >= 0` — luôn đúng.
- `k = 1`: yêu cầu `j - i >= 1` — luôn đúng vì hai vị trí khác nhau thì cách nhau ít nhất 1.

Vậy với `k <= 1`, **chính chuỗi `s` ban đầu** đã là một đáp án. Ta sẽ dùng nhận xét này làm "lối tắt" ở cách 4.

## 3. Kiến thức nền: `Counter`, từ điển `last_pos` và hàm kiểm tra hợp lệ

### 3.1. `collections.Counter` — đếm số lần xuất hiện

`Counter` là một loại `dict` đặc biệt: **key** là phần tử, **value** là số lần xuất hiện. Ta dùng nó để biết mỗi chữ có bao nhiêu "lượt" cần xếp.

- `Counter("aabbcc")` → `{'a': 2, 'b': 2, 'c': 2}`
- `cnt.most_common(1)` → chữ xuất hiện nhiều nhất cùng số lần.
- Hai `Counter` so sánh bằng `==` khi **mọi** chữ có cùng số lần — đây chính là cách kiểm tra "output có đúng là một cách *sắp xếp lại* của `s`" (không thêm, không bớt chữ nào).
- Đọc một key không tồn tại trả về `0` (không báo lỗi như `dict` thường).

In [2]:
from collections import Counter

cnt = Counter("aaadbbcc")
print(cnt)                         # đếm từng chữ
print(cnt["a"], cnt["z"])          # 'z' không có -> 0, không lỗi
print(cnt.most_common(1))          # chữ nhiều nhất
print(max(cnt.values()))           # số lần lớn nhất (maxf)

print(Counter("abacabcd") == Counter("aaadbbcc"))   # cùng bộ chữ -> True
print(Counter("abacabce") == Counter("aaadbbcc"))   # 'e' thay 'd' -> False

Counter({'a': 3, 'b': 2, 'c': 2, 'd': 1})
3 0
[('a', 3)]
3
True
False


### 3.2. Từ điển `last_pos` — nhớ vị trí gần nhất của mỗi chữ

Để biết một chữ đặt ở vị trí `i` có vi phạm hay không, ta chỉ cần biết **lần gần nhất** chữ đó xuất hiện ở đâu. Duyệt chuỗi từ trái sang phải và cập nhật `last_pos[c] = i` mỗi khi gặp `c`:

- Trước khi cập nhật, nếu `c` đã có trong `last_pos` thì khoảng cách là `i - last_pos[c]`.
- Sau đó ghi đè `last_pos[c] = i` (vị trí cũ hơn không còn quan trọng nữa).

Mẹo này sẽ xuất hiện lại dưới tên `next_valid[c] = last_pos[c] + k`: **vị trí sớm nhất** mà `c` được phép xuất hiện lần nữa.

In [3]:
t = "abacabcd"
last_pos = {}
for i, c in enumerate(t):
    if c in last_pos:
        print(f"i={i}: {c!r} lần trước ở {last_pos[c]} -> khoảng cách {i - last_pos[c]}")
    else:
        print(f"i={i}: {c!r} xuất hiện lần đầu")
    last_pos[c] = i
print("last_pos cuối:", last_pos)

i=0: 'a' xuất hiện lần đầu
i=1: 'b' xuất hiện lần đầu
i=2: 'a' lần trước ở 0 -> khoảng cách 2
i=3: 'c' xuất hiện lần đầu
i=4: 'a' lần trước ở 2 -> khoảng cách 2
i=5: 'b' lần trước ở 1 -> khoảng cách 4
i=6: 'c' lần trước ở 3 -> khoảng cách 3
i=7: 'd' xuất hiện lần đầu
last_pos cuối: {'a': 4, 'b': 5, 'c': 6, 'd': 7}


### 3.3. Hàm kiểm tra hợp lệ `is_valid_arrangement(s, k, t)`

Vì đề chấp nhận nhiều đáp án, ta **không** kiểm thử bằng `assert output == "abcabc"`. Thay vào đó, một output `t` được coi là đúng khi thỏa **cả hai** điều kiện:

1. `Counter(t) == Counter(s)` — `t` dùng đúng các chữ của `s`, đúng số lượng.
2. Với mọi chữ, hai lần xuất hiện liên tiếp cách nhau `>= k` (dùng `last_pos`).

Ngoài ra, khi thuật toán trả `""`, ta cần một **trọng tài** khác (brute force ở mục 4) để xác nhận "thật sự không xếp được". Hàm `check_answer` gói cả hai trường hợp lại.

In [4]:
def is_valid_arrangement(s: str, k: int, t: str) -> bool:
    """t có phải một cách sắp xếp lại s với khoảng cách >= k không?"""
    if Counter(t) != Counter(s):
        return False
    last_pos = {}
    for i, c in enumerate(t):
        if c in last_pos and i - last_pos[c] < k:
            return False
        last_pos[c] = i
    return True

print(is_valid_arrangement("aabbcc", 3, "abcabc"))    # True
print(is_valid_arrangement("aabbcc", 3, "acbacb"))    # True: đáp án khác cũng đúng
print(is_valid_arrangement("aabbcc", 3, "abcacb"))    # False: 'b' ở 1 và 5 ok, nhưng 'c' ở 2 và 4 -> 2 < 3
print(is_valid_arrangement("aabbcc", 3, "abcab"))     # False: thiếu một 'c'
print(is_valid_arrangement("aaadbbcc", 2, "abacabcd"))  # True

True
True
False
False
True


## 4. Cách 1 — Brute force: thử mọi hoán vị

Cách người mới nghĩ ra đầu tiên: **liệt kê mọi cách sắp xếp** các chữ của `s`, cách nào hợp lệ thì trả về. Nếu thử hết mà không có cách nào, trả `""`.

### Công cụ: `itertools.permutations`

`permutations(s)` sinh ra mọi **hoán vị** (permutation — mọi thứ tự có thể) của các phần tử, mỗi hoán vị là một **tuple**. Lưu ý: nếu `s` có chữ lặp, nhiều hoán vị bị **trùng** nhau (đổi chỗ hai chữ `a` cho nhau vẫn ra cùng chuỗi). Ta bỏ trùng bằng `set` và dùng `"".join(p)` để ghép tuple thành chuỗi.

In [5]:
from itertools import permutations

all_perms = list(permutations("aab"))
print(len(all_perms), all_perms)            # 3! = 6 hoán vị, có trùng
distinct = sorted(set("".join(p) for p in all_perms))
print(len(distinct), distinct)              # chỉ 3 chuỗi khác nhau

6 [('a', 'a', 'b'), ('a', 'b', 'a'), ('a', 'a', 'b'), ('a', 'b', 'a'), ('b', 'a', 'a'), ('b', 'a', 'a')]
3 ['aab', 'aba', 'baa']


### Code cách 1

Ta duyệt các chuỗi khác nhau theo **thứ tự từ điển** (`sorted`) để kết quả ổn định, dễ đọc. Hàm `rearrange_bruteforce` chỉ dùng cho chuỗi ngắn (khoảng `n <= 8`): nó sẽ làm **trọng tài** quyết định khả thi cho mọi bộ test ngẫu nhiên phía sau.

In [6]:
def rearrange_bruteforce(s: str, k: int) -> str:
    for t in sorted(set("".join(p) for p in permutations(s))):
        if is_valid_arrangement(s, k, t):
            return t
    return ""

for s, k in [("aabbcc", 3), ("aaabc", 3), ("aaadbbcc", 2), ("abb", 2), ("aa", 0)]:
    print(f"s={s!r:11} k={k} -> {rearrange_bruteforce(s, k)!r}")

s='aabbcc'    k=3 -> 'abcabc'
s='aaabc'     k=3 -> ''
s='aaadbbcc'  k=2 -> 'ababacdc'
s='abb'       k=2 -> 'bab'
s='aa'        k=0 -> 'aa'


### Độ phức tạp cách 1 và chỗ vướng

- Số hoán vị là `n!` (có trùng). Mỗi hoán vị tốn `O(n)` để ghép chuỗi và kiểm tra. Tổng: **`O(n! · n)`** thời gian, và `set` giữ tới số chuỗi khác nhau `n! / (c1! · c2! · …)` (công thức **đa thức** — multinomial, `c_i` là số lần chữ thứ `i`).
- `n = 8` → `8! = 40 320` hoán vị: vẫn chạy được trong tích tắc. `n = 15` → hơn `10^12`: bó tay. Mà constraints cho `n` tới `3 · 10^5`.

Cell dưới in số hoán vị khác nhau cho một vài chuỗi để thấy tốc độ "nổ" của nó.

**Chỗ vướng:** brute force thử mù quáng, không tận dụng cấu trúc của bài. Ta cần một cách **xây dần từng vị trí** và ở mỗi vị trí đưa ra quyết định thông minh — đó là **tham lam** (greedy).

In [7]:
from math import factorial

def count_distinct_perms(s: str) -> int:
    total = factorial(len(s))
    for c in Counter(s).values():
        total //= factorial(c)
    return total

for s in ["aabbcc", "aaadbbcc", "aabbccddee", "abcdefghijklmno", "aabbccddeeffgghhiijj"]:
    print(f"n={len(s):2}  {s!r:24} -> {count_distinct_perms(s):,} chuỗi khác nhau")

n= 6  'aabbcc'                 -> 90 chuỗi khác nhau
n= 8  'aaadbbcc'               -> 1,680 chuỗi khác nhau
n=10  'aabbccddee'             -> 113,400 chuỗi khác nhau
n=15  'abcdefghijklmno'        -> 1,307,674,368,000 chuỗi khác nhau
n=20  'aabbccddeeffgghhiijj'   -> 2,375,880,867,360,000 chuỗi khác nhau


## 5. Bước đột phá: xếp từ trái sang phải, mỗi vị trí chọn một chữ "đang được phép"

Hãy thử xếp tay `"aaadbbcc"`, `k = 2` như một người bình thường: điền vị trí `0`, rồi `1`, rồi `2`... Ở mỗi vị trí, chữ nào được phép đặt?

- Một chữ `c` **bị khóa** nếu nó vừa được đặt cách đây chưa đủ `k` vị trí.
- Gọi `next_valid[c]` = **vị trí sớm nhất** `c` được đặt lại. Ban đầu `next_valid[c] = 0`; mỗi khi đặt `c` ở vị trí `i` thì `next_valid[c] = i + k` (đúng bằng `last_pos[c] + k` ở mục 3.2).
- Tại vị trí `i`, chữ `c` **được phép** khi: còn lượt (`count[c] > 0`) **và** `next_valid[c] <= i`.

Câu hỏi then chốt: **trong số các chữ được phép, chọn chữ nào?**

### Cách 2a — Tham lam ngây thơ: chọn chữ nhỏ nhất theo bảng chữ cái

Thử quy tắc đơn giản nhất: lấy chữ được phép **đứng đầu bảng chữ cái**. Nếu không có chữ nào được phép thì bó tay, trả `""`.

In [8]:
def rearrange_greedy_first(s: str, k: int) -> str:
    """Tham lam NGÂY THƠ: chọn chữ được phép nhỏ nhất theo alphabet. (SAI!)"""
    count = Counter(s)
    next_valid = {c: 0 for c in count}
    result = []
    for i in range(len(s)):
        chosen = None
        for c in sorted(count):
            if count[c] > 0 and next_valid[c] <= i:
                chosen = c
                break
        if chosen is None:
            return ""
        result.append(chosen)
        count[chosen] -= 1
        next_valid[chosen] = i + k
    return "".join(result)

print(repr(rearrange_greedy_first("aabbcc", 3)))   # may mắn đúng
print(repr(rearrange_greedy_first("abb", 2)))      # trả "" ...
print(repr(rearrange_bruteforce("abb", 2)))        # ... nhưng "bab" là đáp án!

'abcabc'
''
'bab'


### Vì sao tham lam ngây thơ hỏng? Chữ nhiều nhất là **nút cổ chai**

Với `"abb"`, `k = 2`: ngây thơ đặt `a` ở vị trí 0 (vì `a` đứng đầu alphabet), rồi `b` ở 1, và tới vị trí 2 chỉ còn `b` — nhưng `b` bị khóa tới vị trí 3. Kẹt!

Đúng ra phải đặt `b` **trước**: `b a b`. Lý do sâu xa: một chữ xuất hiện `m` lần cần một "đoạn" dài ít nhất

$$(m - 1)\cdot k + 1$$

vị trí (lần đầu ở `p`, lần cuối ít nhất ở `p + (m-1)·k`). Chữ càng nhiều lần thì càng "tốn chỗ", nên càng phải **bắt đầu sớm** và **không được trì hoãn**. Chữ ít lần (như `a`) thì nhét vào chỗ trống nào cũng được.

**Quy tắc tham lam đúng:** tại mỗi vị trí, trong số các chữ **đang được phép**, chọn chữ **còn nhiều lượt nhất**. Nếu nhiều chữ cùng nhiều nhất, chọn chữ nhỏ nhất theo alphabet (chỉ để kết quả ổn định; ở mục 13 ta sẽ thấy chọn chữ nào trong nhóm hòa cũng không ảnh hưởng tới việc xếp được hay không).

Cell dưới tính "đoạn tối thiểu" của chữ nhiều nhất cho vài ví dụ, so với độ dài chuỗi.

In [9]:
for s, k in [("abb", 2), ("aabbcc", 3), ("aaabc", 3), ("aaadbbcc", 2)]:
    ch, m = Counter(s).most_common(1)[0]
    span = (m - 1) * k + 1
    print(f"s={s!r:11} k={k}: chữ nhiều nhất {ch!r} x{m} cần đoạn >= {span}, chuỗi dài {len(s)}"
          f" -> {'còn chỗ' if span <= len(s) else 'KHÔNG đủ chỗ'}")

s='abb'       k=2: chữ nhiều nhất 'b' x2 cần đoạn >= 3, chuỗi dài 3 -> còn chỗ
s='aabbcc'    k=3: chữ nhiều nhất 'a' x2 cần đoạn >= 4, chuỗi dài 6 -> còn chỗ
s='aaabc'     k=3: chữ nhiều nhất 'a' x3 cần đoạn >= 7, chuỗi dài 5 -> KHÔNG đủ chỗ
s='aaadbbcc'  k=2: chữ nhiều nhất 'a' x3 cần đoạn >= 5, chuỗi dài 8 -> còn chỗ


## 6. Cách 2 — Tham lam quét 26 chữ: chọn chữ còn nhiều nhất đang được phép

### Kiến thức nền: `ord`, `chr` và mảng đếm 26 phần tử

Vì `s` chỉ gồm chữ thường `a..z`, ta có thể thay `dict` bằng **mảng (list) 26 ô**: ô `0` cho `a`, ô `1` cho `b`, …, ô `25` cho `z`.

- `ord(c)` trả về **mã số** (code point) của ký tự: `ord('a') = 97`, `ord('b') = 98`, …
- Vậy `ord(c) - ord('a')` (hay `ord(c) - 97`) đổi chữ thành chỉ số `0..25`.
- `chr(97 + j)` làm ngược lại: chỉ số `j` → chữ.

Dùng mảng có lợi: duyệt "mọi chữ" là một vòng `for j in range(26)` đơn giản, thứ tự luôn là alphabet.

In [10]:
print(ord("a"), ord("b"), ord("z"))          # 97 98 122
print(ord("d") - ord("a"))                    # 3 -> 'd' nằm ở ô số 3
print(chr(97 + 3), chr(97 + 25))              # 'd' 'z'

count = [0] * 26
for c in "aaadbbcc":
    count[ord(c) - 97] += 1
print(count[:5])                              # [3, 2, 2, 1, 0] = a, b, c, d, e
print({chr(97 + j): count[j] for j in range(26) if count[j] > 0})

97 98 122
3
d z
[3, 2, 2, 1, 0]
{'a': 3, 'b': 2, 'c': 2, 'd': 1}


### Thuật toán cách 2 từng bước

Chuẩn bị:

- `count[j]`: số lượt còn lại của chữ `j`.
- `next_valid[j]`: vị trí sớm nhất chữ `j` được đặt lại (ban đầu toàn `0`).

Với mỗi vị trí `i = 0, 1, …, n - 1`:

1. **Quét 26 chữ**, tìm `best` = chữ có `count` **lớn nhất** trong các chữ thỏa `count[j] > 0` và `next_valid[j] <= i`. Vì so sánh `>` (lớn hơn **hẳn**) và quét theo alphabet, khi hòa ta giữ chữ nhỏ nhất.
2. Nếu không có chữ nào (`best == -1`) → mọi chữ còn lượt đều đang bị khóa → **trả `""`**.
3. Đặt chữ `best` vào vị trí `i`: `count[best] -= 1`, `next_valid[best] = i + k`.

Khi điền xong đủ `n` vị trí thì trả kết quả. Mọi chữ đặt xuống đều đã nghỉ đủ, nên output **luôn hợp lệ** nếu khác rỗng.

In [11]:
def rearrange_greedy_scan(s: str, k: int) -> str:
    n = len(s)
    count = [0] * 26
    for c in s:
        count[ord(c) - 97] += 1
    next_valid = [0] * 26
    result = []
    for i in range(n):
        best = -1
        for j in range(26):
            if count[j] > 0 and next_valid[j] <= i:
                if best == -1 or count[j] > count[best]:
                    best = j
        if best == -1:                # mọi chữ còn lượt đều đang bị khóa
            return ""
        result.append(chr(97 + best))
        count[best] -= 1
        next_valid[best] = i + k
    return "".join(result)

for s, k in [("aabbcc", 3), ("aaabc", 3), ("aaadbbcc", 2), ("abb", 2)]:
    out = rearrange_greedy_scan(s, k)
    print(f"s={s!r:11} k={k} -> {out!r:12} hợp lệ? {is_valid_arrangement(s, k, out) if out else '(rỗng)'}")

s='aabbcc'    k=3 -> 'abcabc'     hợp lệ? True
s='aaabc'     k=3 -> ''           hợp lệ? (rỗng)
s='aaadbbcc'  k=2 -> 'abacabcd'   hợp lệ? True
s='abb'       k=2 -> 'bab'        hợp lệ? True


### Trace cách 2 trên Example 3: `s = "aaadbbcc"`, `k = 2`

Ý nghĩa các cột:

- **còn lại**: số lượt mỗi chữ trước khi xét vị trí `i`.
- **bị khóa**: chữ còn lượt nhưng `next_valid > i`; ghi `x(≥p)` nghĩa là `x` phải chờ tới vị trí `p`.
- **được phép**: ứng viên; ta chọn chữ có số lớn nhất (hòa thì chữ nhỏ hơn).

| i | còn lại | bị khóa | được phép | chọn | kết quả |
|---:|---|---|---|---|---|
| 0 | `a3 b2 c2 d1` | `—` | `a3 b2 c2 d1` | **`a`** (đặt lại được từ 2) | `a` |
| 1 | `a2 b2 c2 d1` | `a(≥2)` | `b2 c2 d1` | **`b`** (đặt lại được từ 3) | `ab` |
| 2 | `a2 b1 c2 d1` | `b(≥3)` | `a2 c2 d1` | **`a`** (đặt lại được từ 4) | `aba` |
| 3 | `a1 b1 c2 d1` | `a(≥4)` | `b1 c2 d1` | **`c`** (đặt lại được từ 5) | `abac` |
| 4 | `a1 b1 c1 d1` | `c(≥5)` | `a1 b1 d1` | **`a`** (đặt lại được từ 6) | `abaca` |
| 5 | `b1 c1 d1` | `—` | `b1 c1 d1` | **`b`** (đặt lại được từ 7) | `abacab` |
| 6 | `c1 d1` | `—` | `c1 d1` | **`c`** (đặt lại được từ 8) | `abacabc` |
| 7 | `d1` | `—` | `d1` | **`d`** (đặt lại được từ 9) | `abacabcd` |

Kết quả `"abacabcd"` — trùng khớp với output mẫu của đề. Hãy để ý vị trí `2`: `a` (còn 2) và `c` (còn 2) hòa nhau, ta chọn `a` vì đứng trước trong alphabet; `b` bị khóa vì vừa đặt ở vị trí `1`.

Hàm `trace_greedy_scan` dưới in đúng các trạng thái này.

In [12]:
def trace_greedy_scan(s: str, k: int) -> str:
    count = [0] * 26
    for c in s:
        count[ord(c) - 97] += 1
    next_valid = [0] * 26
    result = []
    for i in range(len(s)):
        allowed = [j for j in range(26) if count[j] > 0 and next_valid[j] <= i]
        locked = [j for j in range(26) if count[j] > 0 and next_valid[j] > i]
        a_str = " ".join(f"{chr(97 + j)}{count[j]}" for j in allowed) or "-"
        l_str = " ".join(f"{chr(97 + j)}(>={next_valid[j]})" for j in locked) or "-"
        if not allowed:
            print(f"i={i}: được phép=- | bị khóa={l_str} -> KẸT, trả ''")
            return ""
        best = max(allowed, key=lambda j: (count[j], -j))   # nhiều nhất, hòa thì chữ nhỏ hơn
        result.append(chr(97 + best))
        count[best] -= 1
        next_valid[best] = i + k
        print(f"i={i}: được phép={a_str:12} | bị khóa={l_str:16} -> chọn {chr(97 + best)!r}, kq={''.join(result)!r}")
    return "".join(result)

trace_greedy_scan("aaadbbcc", 2)

i=0: được phép=a3 b2 c2 d1  | bị khóa=-                -> chọn 'a', kq='a'
i=1: được phép=b2 c2 d1     | bị khóa=a(>=2)           -> chọn 'b', kq='ab'
i=2: được phép=a2 c2 d1     | bị khóa=b(>=3)           -> chọn 'a', kq='aba'
i=3: được phép=b1 c2 d1     | bị khóa=a(>=4)           -> chọn 'c', kq='abac'
i=4: được phép=a1 b1 d1     | bị khóa=c(>=5)           -> chọn 'a', kq='abaca'
i=5: được phép=b1 c1 d1     | bị khóa=-                -> chọn 'b', kq='abacab'
i=6: được phép=c1 d1        | bị khóa=-                -> chọn 'c', kq='abacabc'
i=7: được phép=d1           | bị khóa=-                -> chọn 'd', kq='abacabcd'


'abacabcd'

### Trace khi thất bại: `s = "aaabc"`, `k = 3`

| i | còn lại | bị khóa | được phép | chọn | kết quả |
|---:|---|---|---|---|---|
| 0 | `a3 b1 c1` | `—` | `a3 b1 c1` | **`a`** (đặt lại được từ 3) | `a` |
| 1 | `a2 b1 c1` | `a(≥3)` | `b1 c1` | **`b`** (đặt lại được từ 4) | `ab` |
| 2 | `a2 c1` | `a(≥3)` | `c1` | **`c`** (đặt lại được từ 5) | `abc` |
| 3 | `a2` | `—` | `a2` | **`a`** (đặt lại được từ 6) | `abca` |
| 4 | `a1` | `a(≥6)` | — | **không có** → trả `""` | `abca` |

Tại vị trí `4`, chữ duy nhất còn lượt là `a`, nhưng `a` vừa đặt ở vị trí `3` nên bị khóa tới vị trí `6`. Không có ứng viên → trả `""`. Đây đúng là đáp án của Example 2.

In [13]:
print(repr(trace_greedy_scan("aaabc", 3)))

i=0: được phép=a3 b1 c1     | bị khóa=-                -> chọn 'a', kq='a'
i=1: được phép=b1 c1        | bị khóa=a(>=3)           -> chọn 'b', kq='ab'
i=2: được phép=c1           | bị khóa=a(>=3)           -> chọn 'c', kq='abc'
i=3: được phép=a2           | bị khóa=-                -> chọn 'a', kq='abca'
i=4: được phép=- | bị khóa=a(>=6) -> KẸT, trả ''
''


### Độ phức tạp cách 2 và chỗ vướng

- **Time:** `n` vị trí × quét 26 chữ = **`O(26 · n)`**. Với `n = 3·10^5` là khoảng `7,8 triệu` phép kiểm tra — chạy được, nhưng trong Python chậm hơn cách 3 nhiều lần (mục 12 đo thử).
- **Space:** hai mảng 26 ô → `O(26) = O(1)` phụ trợ, cộng `O(n)` cho chuỗi kết quả.

**Chỗ vướng:**

1. Ở mỗi vị trí ta quét lại **toàn bộ 26 chữ**, kể cả những chữ đã hết lượt hoặc đang bị khóa. Nếu bảng chữ lớn hơn (ví dụ bài #1054 với barcode là số nguyên tới `10^4`), `O(σ · n)` trở nên quá chậm.
2. Thật ra ta chỉ cần hai thao tác nhanh: **(a)** lấy ra chữ **còn nhiều nhất** trong nhóm "được phép"; **(b)** biết chữ nào **hết bị khóa** đúng ở vị trí hiện tại.

Thao tác (a) là việc của **heap** (đống ưu tiên). Thao tác (b) có một quan sát đẹp: chữ bị khóa theo thứ tự **vào trước — ra trước** (chữ đặt sớm hơn thì hết khóa sớm hơn), nên ta giữ chúng trong một **hàng đợi** (queue). Trước khi ghép lại, hãy học hai cấu trúc này từ đầu.

## 7. Kiến thức nền: heap với `heapq`

**Heap** (đống) là một cấu trúc giúp lấy ra **phần tử nhỏ nhất** rất nhanh. Python có module `heapq` thao tác trên một `list` thường:

| Lệnh | Ý nghĩa | Thời gian |
|---|---|---|
| `heapq.heapify(h)` | biến list `h` thành heap | `O(len)` |
| `heapq.heappush(h, x)` | thêm `x` | `O(log len)` |
| `heapq.heappop(h)` | lấy ra **và xóa** phần tử nhỏ nhất | `O(log len)` |
| `h[0]` | xem phần tử nhỏ nhất (không xóa) | `O(1)` |

Bên trong, heap là một cây nhị phân "gần đầy" lưu trong list, với tính chất: **cha luôn `<=` con**. Nhờ vậy phần tử nhỏ nhất luôn nằm ở `h[0]`, và mỗi lần thêm/xóa chỉ cần sửa dọc **một nhánh** cao `log len`. Lưu ý: list của heap **không** được sắp xếp hoàn toàn — chỉ chắc chắn `h[0]` là nhỏ nhất.

In [14]:
import heapq

h = [5, 1, 8, 3, 2]
heapq.heapify(h)
print("sau heapify:", h, "-> nhỏ nhất h[0] =", h[0])   # không sắp xếp hoàn toàn!
heapq.heappush(h, 0)
print("push 0     :", h)
print("pop lần lượt:", [heapq.heappop(h) for _ in range(len(h))])   # ra theo thứ tự tăng dần

sau heapify: [1, 2, 8, 3, 5] -> nhỏ nhất h[0] = 1
push 0     : [0, 2, 1, 3, 5, 8]
pop lần lượt: [0, 1, 2, 3, 5, 8]


### Max-heap bằng mẹo **đổi dấu** và tuple `(-count, chữ)`

`heapq` chỉ có **min-heap**. Muốn lấy chữ **còn nhiều nhất**, ta lưu `-count`: số càng lớn thì `-count` càng nhỏ, nên nằm ở đỉnh.

Ta lưu **tuple** `(-count, chữ)`. Python so sánh tuple **từ trái sang phải**: so phần tử đầu trước; nếu bằng nhau mới so phần tử thứ hai. Do đó:

- `(-3, 'a') < (-2, 'b')` → `a` (còn 3) ra trước `b` (còn 2).
- `(-2, 'a') < (-2, 'c')` → hòa số lượt thì chữ nhỏ hơn theo alphabet ra trước — **đúng quy tắc hòa** của cách 2!

Khi lấy ra phải **đổi dấu lại** để có số lượt thật.

In [15]:
cnt = Counter("aaadbbcc")
heap = [(-v, c) for c, v in cnt.items()]
heapq.heapify(heap)
print("heap:", heap)

order = []
while heap:
    neg, c = heapq.heappop(heap)
    order.append((c, -neg))        # đổi dấu lại
print("thứ tự lấy ra:", order)      # a(3) trước, rồi b(2) trước c(2), cuối cùng d(1)
print((-2, "a") < (-2, "c"), (-3, "z") < (-2, "a"))

heap: [(-3, 'a'), (-2, 'c'), (-2, 'b'), (-1, 'd')]
thứ tự lấy ra: [('a', 3), ('b', 2), ('c', 2), ('d', 1)]
True True


## 8. Kiến thức nền: hàng đợi FIFO với `collections.deque`

**Queue** (hàng đợi) hoạt động như xếp hàng mua vé: ai vào trước thì ra trước — **FIFO** (First In, First Out).

`collections.deque` (đọc là "deck", double-ended queue) cho phép thêm/bớt ở **cả hai đầu** trong `O(1)`:

- `q.append(x)`: thêm vào **cuối** hàng.
- `q.popleft()`: lấy ra người ở **đầu** hàng.
- `len(q)`: số người đang chờ.

Đừng dùng `list.pop(0)` thay cho `popleft()`: nó phải dời mọi phần tử còn lại lên một ô nên tốn `O(len)`.

Trong bài này, hàng đợi đóng vai **"phòng chờ"** (cooldown): chữ vừa đặt xuống vào cuối hàng, và chỉ khi nó lên tới đầu hàng (sau đủ lượt) mới được trở lại vòng tuyển chọn.

In [16]:
from collections import deque

q = deque()
for c in "abc":
    q.append(c)
    print("append", c, "->", list(q))
print("popleft ->", q.popleft(), "| còn", list(q))   # 'a' vào trước nên ra trước
q.append("a")
print("append a ->", list(q), "| len =", len(q))

append a -> ['a']
append b -> ['a', 'b']
append c -> ['a', 'b', 'c']
popleft -> a | còn ['b', 'c']
append a -> ['b', 'c', 'a'] | len = 3


## 9. Cách 3 (cách chính) — Max-heap + hàng đợi chờ kích thước `k`

### Ý tưởng: hai "khu vực" cho các chữ

Mỗi chữ còn liên quan tới bài toán luôn nằm ở **đúng một** trong hai khu vực:

1. **Heap "sẵn sàng"** — các chữ còn lượt và **đang được phép** đặt. Đỉnh heap là chữ còn nhiều lượt nhất (hòa thì chữ nhỏ nhất).
2. **Hàng đợi "phòng chờ"** — các chữ vừa đặt ở `k - 1` vị trí gần nhất, xếp theo thứ tự đã đặt.

Mỗi bước (mỗi vị trí):

1. Lấy chữ ở đỉnh heap, nối vào kết quả, giảm số lượt đi 1.
2. Đưa chữ đó (kèm số lượt **còn lại**, **kể cả khi đã về 0**) vào **cuối** phòng chờ.
3. Nếu phòng chờ đã có `>= k` chữ: lấy chữ ở **đầu** phòng chờ ra; nếu nó còn lượt thì trả về heap, nếu hết lượt thì bỏ đi.

Vòng lặp chạy khi heap còn chữ. Khi heap rỗng mà kết quả chưa đủ `n` ký tự → còn chữ mắc kẹt trong phòng chờ, không xếp tiếp được → trả `""`.

### Vì sao điều kiện là `len(q) >= k` thì khoảng cách đúng bằng `k`?

Giả sử chữ `x` được đặt ở vị trí `p` và vào cuối hàng. Sau đó ở các vị trí `p+1, …, p+k-1` mỗi bước thêm 1 chữ vào cuối hàng. Đúng tại bước `p + k - 1`, hàng có `k` chữ (từ vị trí `p` tới `p+k-1`) và `x` ở đầu → `x` được thả về heap. Vậy sớm nhất `x` được đặt lại ở vị trí `p + k`: khoảng cách **đúng bằng `k`**, không sớm hơn (hợp lệ) mà cũng không muộn hơn (không lãng phí).

### Vì sao phải đưa cả chữ **đã hết lượt** (còn 0) vào phòng chờ?

Phòng chờ giống một **đồng hồ**: "người thứ `k` tính từ cuối" chính là chữ đặt cách đây `k - 1` bước. Nếu bỏ qua chữ hết lượt, hàng đợi sẽ ngắn đi một nhịp và đồng hồ chạy **chậm** — các chữ khác bị giữ lâu hơn cần thiết, dẫn tới trả `""` sai với những chuỗi xếp được. (Mục 15 có ví dụ cụ thể.)

In [17]:
def rearrange_heap_queue(s: str, k: int) -> str:
    if k <= 1:                                  # cách 4: mọi cách xếp đều hợp lệ
        return s
    heap = [(-v, c) for c, v in Counter(s).items()]
    heapq.heapify(heap)
    wait = deque()                              # phòng chờ: (-còn_lại, chữ)
    result = []
    while heap:
        neg, c = heapq.heappop(heap)            # 1. chữ còn nhiều nhất đang được phép
        result.append(c)
        wait.append((neg + 1, c))               # 2. vào phòng chờ (kể cả khi còn 0)
        if len(wait) >= k:                      # 3. chữ đã chờ đủ -> thả
            neg2, c2 = wait.popleft()
            if neg2 < 0:                        #    còn lượt mới trả về heap
                heapq.heappush(heap, (neg2, c2))
    return "".join(result) if len(result) == len(s) else ""

for s, k in [("aabbcc", 3), ("aaabc", 3), ("aaadbbcc", 2), ("abb", 2)]:
    print(f"s={s!r:11} k={k} -> {rearrange_heap_queue(s, k)!r}")

s='aabbcc'    k=3 -> 'abcabc'
s='aaabc'     k=3 -> ''
s='aaadbbcc'  k=2 -> 'abacabcd'
s='abb'       k=2 -> 'bab'


### Trace kỹ `s = "aabbcc"`, `k = 3`

Quy ước hiển thị: `a2` nghĩa là chữ `a` còn 2 lượt. Cột **heap trước** là các chữ đang được phép trước khi lấy; cột **phòng chờ** là hàng đợi **sau khi** thêm chữ vừa đặt (đầu hàng bên trái); cột **bước 3** cho biết có thả ai không.

| i | heap trước | lấy | kết quả | phòng chờ (sau khi thêm) | bước 3 | heap sau |
|---:|---|---|---|---|---|---|
| 0 | `a2 b2 c2` | **`a`** | `a` | `a1` | chưa thả (1 < 3) | `b2 c2` |
| 1 | `b2 c2` | **`b`** | `ab` | `a1, b1` | chưa thả (2 < 3) | `c2` |
| 2 | `c2` | **`c`** | `abc` | `a1, b1, c1` | thả `a` (còn 1) về heap | `a1` |
| 3 | `a1` | **`a`** | `abca` | `b1, c1, a0` | thả `b` (còn 1) về heap | `b1` |
| 4 | `b1` | **`b`** | `abcab` | `c1, a0, b0` | thả `c` (còn 1) về heap | `c1` |
| 5 | `c1` | **`c`** | `abcabc` | `a0, b0, c0` | bỏ `a` (hết lượt) | `(rỗng)` |

Những điểm cần nhìn kỹ:

- **Bước `i = 1, 2`**: heap chỉ còn `b2 c2` rồi `c2`, vì `a` (rồi `b`) đang ngồi trong phòng chờ. Dù `a` còn 2 lượt — nhiều nhất — nó **không được phép** chọn lại ngay. Đây là điểm khác với "cứ lấy max".
- **Bước `i = 2`**: phòng chờ đủ 3 chữ → `a` (đặt ở vị trí 0) được thả, kịp để đặt ở vị trí `3`: khoảng cách đúng `3`.
- **Bước `i = 5`**: `a0` ra khỏi phòng chờ nhưng **hết lượt** nên bị bỏ, không trả về heap. Heap rỗng → vòng lặp dừng, kết quả đủ 6 ký tự.

In [18]:
def fmt_heap(heap):
    return " ".join(f"{c}{-v}" for v, c in sorted(heap)) or "(rỗng)"

def fmt_queue(q):
    return ", ".join(f"{c}{-v}" for v, c in q) or "(rỗng)"

def trace_heap_queue(s: str, k: int) -> str:
    heap = [(-v, c) for c, v in Counter(s).items()]
    heapq.heapify(heap)
    wait, result, i = deque(), [], 0
    while heap:
        before = fmt_heap(heap)
        neg, c = heapq.heappop(heap)
        result.append(c)
        wait.append((neg + 1, c))
        line = f"i={i}: heap=[{before}] lấy {c!r} kq={''.join(result)!r:9} chờ=[{fmt_queue(wait)}]"
        if len(wait) >= k:
            neg2, c2 = wait.popleft()
            if neg2 < 0:
                heapq.heappush(heap, (neg2, c2))
                line += f" -> thả {c2}{-neg2} về heap"
            else:
                line += f" -> bỏ {c2} (hết lượt)"
        else:
            line += f" -> chưa thả ({len(wait)}<{k})"
        print(line)
        i += 1
    ans = "".join(result) if len(result) == len(s) else ""
    print(f"heap rỗng sau {len(result)}/{len(s)} ký tự -> trả {ans!r}")
    return ans

trace_heap_queue("aabbcc", 3)

i=0: heap=[a2 b2 c2] lấy 'a' kq='a'       chờ=[a1] -> chưa thả (1<3)
i=1: heap=[b2 c2] lấy 'b' kq='ab'      chờ=[a1, b1] -> chưa thả (2<3)
i=2: heap=[c2] lấy 'c' kq='abc'     chờ=[a1, b1, c1] -> thả a1 về heap
i=3: heap=[a1] lấy 'a' kq='abca'    chờ=[b1, c1, a0] -> thả b1 về heap
i=4: heap=[b1] lấy 'b' kq='abcab'   chờ=[c1, a0, b0] -> thả c1 về heap
i=5: heap=[c1] lấy 'c' kq='abcabc'  chờ=[a0, b0, c0] -> bỏ a (hết lượt)
heap rỗng sau 6/6 ký tự -> trả 'abcabc'


'abcabc'

### Trace khi thất bại: `s = "aaabc"`, `k = 3`

Lần này hãy dự đoán trước: `a` cần đặt 3 lần nhưng chỉ có `b`, `c` để chen vào giữa. Theo dõi cột phòng chờ: ở bước `i = 3`, sau khi đặt `a` lần hai, heap **rỗng** trong khi `a` còn 1 lượt đang **mắc kẹt** trong phòng chờ. Vòng lặp dừng với 4/5 ký tự → trả `""`.

Đây chính là lý do dòng cuối phải kiểm tra `len(result) == len(s)`: nếu quên, hàm sẽ trả `"abca"` — một chuỗi **thiếu chữ**.

In [19]:
trace_heap_queue("aaabc", 3)
print()
trace_heap_queue("aaadbbcc", 2)    # Example 3: ra đúng "abacabcd"

i=0: heap=[a3 b1 c1] lấy 'a' kq='a'       chờ=[a2] -> chưa thả (1<3)
i=1: heap=[b1 c1] lấy 'b' kq='ab'      chờ=[a2, b0] -> chưa thả (2<3)
i=2: heap=[c1] lấy 'c' kq='abc'     chờ=[a2, b0, c0] -> thả a2 về heap
i=3: heap=[a2] lấy 'a' kq='abca'    chờ=[b0, c0, a1] -> bỏ b (hết lượt)
heap rỗng sau 4/5 ký tự -> trả ''

i=0: heap=[a3 b2 c2 d1] lấy 'a' kq='a'       chờ=[a2] -> chưa thả (1<2)
i=1: heap=[b2 c2 d1] lấy 'b' kq='ab'      chờ=[a2, b1] -> thả a2 về heap
i=2: heap=[a2 c2 d1] lấy 'a' kq='aba'     chờ=[b1, a1] -> thả b1 về heap
i=3: heap=[c2 b1 d1] lấy 'c' kq='abac'    chờ=[a1, c1] -> thả a1 về heap
i=4: heap=[a1 b1 d1] lấy 'a' kq='abaca'   chờ=[c1, a0] -> thả c1 về heap
i=5: heap=[b1 c1 d1] lấy 'b' kq='abacab'  chờ=[a0, b0] -> bỏ a (hết lượt)
i=6: heap=[c1 d1] lấy 'c' kq='abacabc' chờ=[b0, c0] -> bỏ b (hết lượt)
i=7: heap=[d1] lấy 'd' kq='abacabcd' chờ=[c0, d0] -> bỏ c (hết lượt)
heap rỗng sau 8/8 ký tự -> trả 'abacabcd'


'abacabcd'

## 10. Cách 4 — Xử lý `k = 0` và `k = 1`

Như mục 2 đã nói, `k <= 1` thì điều kiện "khoảng cách `>= k`" **luôn đúng**, nên trả lại chính `s` là một đáp án hợp lệ trong `O(1)` (không cần sắp xếp gì).

Vì sao vẫn nên viết lối tắt này dù thuật toán heap chạy đúng cả với `k = 0, 1`?

- **Hiệu năng:** bỏ qua `n` lần thao tác heap.
- **An toàn:** nhiều bản cài đặt dùng điều kiện `len(wait) == k` (thay vì `>= k`). Với `k = 0`, sau khi `append` hàng luôn có ít nhất 1 chữ nên `len(wait) == 0` **không bao giờ đúng** → không ai được thả → trả `""` sai! Lối tắt `if k <= 1: return s` chặn luôn lỗi này. (Ví dụ cụ thể ở mục 15.)

Cell dưới so sánh hàm có lối tắt với bản heap "không lối tắt" (điều kiện `>=`) cho `k = 0, 1`.

In [20]:
def rearrange_heap_queue_no_shortcut(s: str, k: int) -> str:
    heap = [(-v, c) for c, v in Counter(s).items()]
    heapq.heapify(heap)
    wait, result = deque(), []
    while heap:
        neg, c = heapq.heappop(heap)
        result.append(c)
        wait.append((neg + 1, c))
        if len(wait) >= k:
            neg2, c2 = wait.popleft()
            if neg2 < 0:
                heapq.heappush(heap, (neg2, c2))
    return "".join(result) if len(result) == len(s) else ""

for s, k in [("aaab", 0), ("aaab", 1), ("zzz", 1)]:
    a = rearrange_heap_queue(s, k)
    b = rearrange_heap_queue_no_shortcut(s, k)
    print(f"s={s!r} k={k}: lối tắt -> {a!r} (hợp lệ {is_valid_arrangement(s, k, a)}), "
          f"không lối tắt -> {b!r} (hợp lệ {is_valid_arrangement(s, k, b)})")

s='aaab' k=0: lối tắt -> 'aaab' (hợp lệ True), không lối tắt -> 'aaab' (hợp lệ True)
s='aaab' k=1: lối tắt -> 'aaab' (hợp lệ True), không lối tắt -> 'aaab' (hợp lệ True)
s='zzz' k=1: lối tắt -> 'zzz' (hợp lệ True), không lối tắt -> 'zzz' (hợp lệ True)


### Cách 2 và cách 3 cho ra **cùng một chuỗi** (khi `k >= 2`)

Hai cách dùng **cùng quy tắc chọn**: trong các chữ được phép, chọn chữ còn nhiều nhất, hòa thì chữ nhỏ nhất. Khác biệt chỉ là *cách tìm* chữ đó:

- Cách 2 quét 26 ô, kiểm tra `next_valid[j] <= i`.
- Cách 3 để heap tự giữ đúng tập "được phép" (chữ bị khóa nằm trong phòng chờ chứ không nằm trong heap).

Vì vậy output phải **trùng từng ký tự**. Cell dưới kiểm chứng trên 2000 chuỗi ngẫu nhiên — đây là một bằng chứng mạnh rằng phòng chờ mô phỏng đúng `next_valid`.

In [21]:
import random

random.seed(358)
for _ in range(2000):
    n = random.randint(2, 12)
    s = "".join(random.choice("abcde") for _ in range(n))
    k = random.randint(2, n)                    # đúng constraints: k <= n
    assert rearrange_greedy_scan(s, k) == rearrange_heap_queue(s, k), (s, k)
print("Cách 2 và cách 3 trùng output trên 2000 chuỗi ngẫu nhiên (k >= 2)")

Cách 2 và cách 3 trùng output trên 2000 chuỗi ngẫu nhiên (k >= 2)


## 11. Bonus — Công thức biết ngay có xếp được hay không

Gọi `maxf` = số lần xuất hiện **lớn nhất**, `cnt_max` = **số chữ** đạt đúng `maxf` lần. Khi đó:

$$\text{xếp được} \iff (maxf - 1)\cdot k + cnt\_max \le n$$

### Hình dung bằng "khung"

Xếp các chữ nhiều nhất theo **hàng**, mỗi hàng dài `k` ô. Với `s = "aabbcc"`, `k = 3`: `maxf = 2`, `cnt_max = 3`:

```text
hàng 0:  a b c        <- (maxf - 1) = 1 hàng đầy, mỗi hàng k = 3 ô
hàng 1:  a b c        <- hàng cuối chỉ chứa cnt_max = 3 chữ nhiều nhất
```

Cần `1·3 + 3 = 6` ô, và `n = 6` → vừa khít, xếp được.

Với `s = "aaabc"`, `k = 3`: `maxf = 3`, `cnt_max = 1` → cần `2·3 + 1 = 7 > 5` → không xếp được.

### Vì sao điều kiện này là **cần**? (chứng minh ngắn)

Xét `cnt_max` chữ cùng xuất hiện `maxf` lần. Lần xuất hiện **đầu tiên** của chúng ở `cnt_max` vị trí khác nhau, nên chữ có lần đầu muộn nhất bắt đầu ở vị trí `>= cnt_max - 1`. Chữ đó cần thêm `(maxf - 1)` bước nhảy, mỗi bước `>= k`, nên lần cuối của nó ở vị trí `>= cnt_max - 1 + (maxf - 1)·k`. Vị trí này phải `<= n - 1`, suy ra `(maxf - 1)·k + cnt_max <= n`.

### Còn chiều **đủ**?

Chiều ngược lại (điều kiện thỏa thì xếp được) đúng là điều mà thuật toán tham lam thực hiện: nó luôn xếp các chữ nhiều nhất "xoay vòng" theo từng hàng như khung trên. Ta không chứng minh chặt chẽ ở đây, nhưng kiểm chứng **vét cạn** với brute force ở cell dưới và ở mục 13.

Công thức này cũng là trái tim của bài **#621 Task Scheduler** (Bài tập 5).

In [22]:
def can_rearrange_formula(s: str, k: int) -> bool:
    cnt = Counter(s)
    maxf = max(cnt.values())
    cnt_max = sum(1 for v in cnt.values() if v == maxf)
    return (maxf - 1) * k + cnt_max <= len(s)

for s, k in [("aabbcc", 3), ("aaabc", 3), ("aaadbbcc", 2), ("abb", 2), ("aabb", 3), ("aaabbbcc", 3)]:
    f = can_rearrange_formula(s, k)
    b = rearrange_bruteforce(s, k) != ""
    print(f"s={s!r:11} k={k}: công thức -> {f!s:5}  brute force -> {b!s:5}  {'khớp' if f == b else 'LỆCH!'}")

s='aabbcc'    k=3: công thức -> True   brute force -> True   khớp
s='aaabc'     k=3: công thức -> False  brute force -> False  khớp
s='aaadbbcc'  k=2: công thức -> True   brute force -> True   khớp
s='abb'       k=2: công thức -> True   brute force -> True   khớp
s='aabb'      k=3: công thức -> False  brute force -> False  khớp
s='aaabbbcc'  k=3: công thức -> True   brute force -> True   khớp


## 12. Code nộp LeetCode

Chuyển cách 3 + cách 4 sang đúng format `class Solution`. So với bản demo, bản nộp:

- Giữ đúng signature `rearrangeString(self, s: str, k: int) -> str`.
- Tự `import` những gì cần (`Counter`, `deque`, `heapq`) — trên LeetCode, `heapq` và `collections` thường đã có sẵn, nhưng import rõ ràng giúp code chạy được ở mọi nơi.
- Không `print` debug.

Kiểm thử bằng `check_answer`: nếu output khác rỗng thì phải **hợp lệ** theo `is_valid_arrangement`; nếu output rỗng thì brute force (chỉ với chuỗi ngắn) hoặc công thức khả thi phải xác nhận là **không xếp được**.

In [23]:
import heapq
from collections import Counter, deque


class Solution:
    def rearrangeString(self, s: str, k: int) -> str:
        if k <= 1:
            return s
        heap = [(-v, c) for c, v in Counter(s).items()]
        heapq.heapify(heap)
        wait = deque()
        result = []
        while heap:
            neg, c = heapq.heappop(heap)
            result.append(c)
            wait.append((neg + 1, c))
            if len(wait) >= k:
                neg2, c2 = wait.popleft()
                if neg2 < 0:
                    heapq.heappush(heap, (neg2, c2))
        return "".join(result) if len(result) == len(s) else ""


def check_answer(s: str, k: int, out: str, possible: bool, name: str = ""):
    if possible:
        assert out != "" and is_valid_arrangement(s, k, out), (name, s, k, out)
    else:
        assert out == "", (name, s, k, out)
    print(f"PASS {name:28} s={s!r:12} k={k} -> {out!r}")


sol = Solution()
tests = [
    ("aabbcc", 3, True, "Example 1"),
    ("aaabc", 3, False, "Example 2"),
    ("aaadbbcc", 2, True, "Example 3"),
    ("a", 0, True, "1 ký tự, k = 0"),
    ("a", 1, True, "1 ký tự, k = 1"),
    ("aa", 2, False, "2 chữ giống, k = 2"),
    ("aaab", 0, True, "k = 0: trả nguyên s"),
    ("abcd", 4, True, "k = n, mọi chữ khác nhau"),
    ("aab", 3, False, "k = n nhưng có chữ lặp"),
    ("abb", 2, True, "bẫy của tham lam ngây thơ"),
    ("aaabbbcc", 3, True, "vừa khít công thức"),
]
for s, k, possible, name in tests:
    check_answer(s, k, sol.rearrangeString(s, k), possible, name)

PASS Example 1                    s='aabbcc'     k=3 -> 'abcabc'
PASS Example 2                    s='aaabc'      k=3 -> ''
PASS Example 3                    s='aaadbbcc'   k=2 -> 'abacabcd'
PASS 1 ký tự, k = 0               s='a'          k=0 -> 'a'
PASS 1 ký tự, k = 1               s='a'          k=1 -> 'a'
PASS 2 chữ giống, k = 2           s='aa'         k=2 -> ''
PASS k = 0: trả nguyên s          s='aaab'       k=0 -> 'aaab'
PASS k = n, mọi chữ khác nhau     s='abcd'       k=4 -> 'abcd'
PASS k = n nhưng có chữ lặp       s='aab'        k=3 -> ''
PASS bẫy của tham lam ngây thơ    s='abb'        k=2 -> 'bab'
PASS vừa khít công thức           s='aaabbbcc'   k=3 -> 'abcabcab'


### Thử với input lớn nhất theo constraints (`n = 3·10^5`)

Ba tình huống:

1. Bảng chữ cái `"abc…z"` lặp lại, `k = 26`: **vừa khít** — mỗi chữ phải xuất hiện đúng mỗi 26 vị trí.
2. Chuỗi ngẫu nhiên 26 chữ, `k = 10`.
3. Toàn chữ `a`, `k = 2`: không thể; heap rỗng ngay sau 1 bước nên kết thúc rất nhanh.

So thời gian cách 3 với cách 2 (quét 26 chữ) để cảm nhận hằng số `26` (máy khác nhau cho số khác nhau, chỉ nên so tỉ lệ).

In [24]:
import time

rng = random.Random(1)
big_cases = [
    ("".join(chr(97 + i % 26) for i in range(300_000)), 26, "alphabet lặp, k=26"),
    ("".join(rng.choice("abcdefghijklmnopqrstuvwxyz") for _ in range(300_000)), 10, "ngẫu nhiên, k=10"),
    ("a" * 300_000, 2, "toàn 'a', k=2"),
]
for s, k, name in big_cases:
    t0 = time.perf_counter(); out3 = sol.rearrangeString(s, k); t1 = time.perf_counter()
    out2 = rearrange_greedy_scan(s, k); t2 = time.perf_counter()
    ok = (out3 == "" and not can_rearrange_formula(s, k)) or is_valid_arrangement(s, k, out3)
    assert ok and out2 == out3
    print(f"{name:20}: len kết quả={len(out3):6}, đúng={ok}, heap {t1 - t0:.2f}s, quét-26 {t2 - t1:.2f}s")

alphabet lặp, k=26  : len kết quả=300000, đúng=True, heap 0.05s, quét-26 0.36s


ngẫu nhiên, k=10    : len kết quả=300000, đúng=True, heap 0.08s, quét-26 0.55s
toàn 'a', k=2       : len kết quả=     0, đúng=True, heap 0.01s, quét-26 0.01s


### Kiểm thử đối chiếu chéo tất cả các cách

Sinh ngẫu nhiên (với `random.seed` cố định để chạy lại ra cùng kết quả) các chuỗi ngắn trên bảng chữ nhỏ (`"ab"`, `"abc"`, `"abcd"` — chữ ít thì dễ có trùng lặp, dễ gặp ca khó), `k` từ `0` tới `n`. Với mỗi input:

- Brute force quyết định **khả thi** hay không.
- Cách 2, cách 3 và `Solution` phải cho kết quả **hợp lệ** khi khả thi, và `""` khi không.
- Công thức ở mục 11 phải khớp brute force.

In [25]:
random.seed(2024)
stats = Counter()
for _ in range(1500):
    n = random.randint(1, 8)
    s = "".join(random.choice(random.choice(["ab", "abc", "abcd"])) for _ in range(n))
    k = random.randint(0, n)
    possible = rearrange_bruteforce(s, k) != ""
    stats["khả thi" if possible else "không khả thi"] += 1
    for name, f in [("greedy_scan", rearrange_greedy_scan),
                    ("heap_queue", rearrange_heap_queue),
                    ("Solution", sol.rearrangeString)]:
        out = f(s, k)
        if possible:
            assert out and is_valid_arrangement(s, k, out), (name, s, k, out)
        else:
            assert out == "", (name, s, k, out)
    assert can_rearrange_formula(s, k) == possible, (s, k)
print("1500 test ngẫu nhiên PASS:", dict(stats))

1500 test ngẫu nhiên PASS: {'không khả thi': 533, 'khả thi': 967}


## 13. Vì sao cách 3 đúng?

Ta tách thành hai câu hỏi: **(A)** output khác rỗng có luôn hợp lệ không? **(B)** khi trả `""`, có chắc là không xếp được không?

### (A) Mọi output khác rỗng đều hợp lệ — invariant "đồng hồ phòng chờ"

**Invariant 1.** Ngay sau bước `i` (đã đặt `i + 1` chữ, với `k >= 2`), phòng chờ chứa **đúng** các chữ đặt ở `min(i + 1, k - 1)` vị trí gần nhất, theo thứ tự đặt.

*Lý do:* mỗi bước thêm đúng 1 chữ vào cuối; khi độ dài chạm `k` thì lấy ra đúng 1 chữ ở đầu, nên độ dài không bao giờ vượt `k - 1` sau bước 3. Chữ được lấy ra là chữ đặt cách đây `k - 1` bước. (Nhớ là chữ hết lượt **vẫn** được thêm vào, nên đếm đúng mọi vị trí.)

**Invariant 2.** Trước mỗi bước, heap chứa **đúng** các chữ còn lượt **và** không nằm trong `k - 1` vị trí gần nhất — tức đúng tập "được phép" của cách 2.

Từ hai invariant: chữ đặt ở vị trí `p` chỉ quay về heap ở cuối bước `p + k - 1`, nên lần đặt kế tiếp có vị trí `>= p + k`. Không cặp nào vi phạm. Thêm nữa, mỗi lần lấy ra là trừ đúng 1 lượt, nên khi `len(result) == n` thì `Counter(result) == Counter(s)`.

### (B) Khi trả `""` thì thật sự không xếp được — vì sao tham lam "chọn nhiều nhất" đúng?

Trực giác: theo mục 11, thứ làm bài toán khó là các chữ **nhiều nhất** — chúng cần khung `(maxf - 1)·k + cnt_max` ô. Tham lam luôn phục vụ chữ còn nhiều nhất **trước**, nên các chữ "nặng" được rải đều theo từng nhịp `k`, đúng như hình khung. Chọn một chữ ít lượt thay vào chỗ đó chỉ làm khung của chữ nặng **dài thêm**, không bao giờ có lợi.

Một mảnh chứng minh chặt chẽ: **chọn chữ nào trong nhóm hòa cũng như nhau.** Giả sử `x` và `y` cùng được phép và còn cùng số lượt. Nếu có một cách xếp hợp lệ đặt `y` tại vị trí `i`, ta **đổi tên** `x ↔ y` trên toàn bộ phần từ `i` trở đi. Số lượt hai chữ bằng nhau nên vẫn đủ chữ; cả hai đều đã "nghỉ đủ" trước vị trí `i` nên ranh giới giữa phần trước và phần sau không vi phạm. Vậy luôn có cách xếp hợp lệ đặt `x` tại `i`.

Chứng minh đầy đủ cho trường hợp số lượt khác nhau dài hơn (dùng lập luận đổi chỗ — exchange argument), nằm ngoài phạm vi notebook nhập môn. Thay vào đó ta **vét cạn**: kiểm tra **mọi** chuỗi (không tính thứ tự) độ dài `1..7` trên bảng `"abcd"` với **mọi** `k` từ `0` tới `n`, so với brute force.

In [26]:
from itertools import combinations_with_replacement

checked = 0
for n in range(1, 8):
    for combo in combinations_with_replacement("abcd", n):   # mỗi multiset chữ đúng 1 lần
        s = "".join(combo)
        for k in range(0, n + 1):
            possible = rearrange_bruteforce(s, k) != ""
            out = rearrange_heap_queue(s, k)
            assert (out != "") == possible, (s, k, out)
            if out:
                assert is_valid_arrangement(s, k, out)
            assert can_rearrange_formula(s, k) == possible
            checked += 1
print(f"Vét cạn {checked} cặp (s, k): tham lam heap + công thức khớp brute force 100%")

Vét cạn 2177 cặp (s, k): tham lam heap + công thức khớp brute force 100%


## 14. Độ phức tạp và so sánh các cách

Ký hiệu: `n = len(s)`, `σ` = số chữ khác nhau (`σ <= 26`).

**Cách 3 (heap + phòng chờ):**

- Mỗi vị trí làm tối đa 1 `heappop` và 1 `heappush` trên heap có **không quá `σ`** phần tử → `O(log σ)`. Tổng **`O(n log σ)`** = `O(n log 26)`, thực chất là `O(n)` với hằng số nhỏ. Dựng `Counter` và `heapify` tốn `O(n + σ)`.
- **Space:** mỗi chữ chỉ nằm ở **một** nơi (heap hoặc phòng chờ), nên heap + phòng chờ cộng lại `<= σ` phần tử → `O(σ)` phụ trợ, cộng `O(n)` cho chuỗi kết quả. Một hệ quả thú vị: phòng chờ không bao giờ dài quá `σ`, nên nếu `k > σ` mà có chữ lặp, phòng chờ không bao giờ đầy `k` → heap cạn → trả `""` (đúng: giữa hai chữ giống nhau cần `k - 1 >= σ` chữ khác nhau, mà chỉ có `σ - 1` chữ khác).

| Cách | Ý tưởng | Time | Space phụ | Ghi chú |
|---|---|---|---|---|
| 1. Brute force | thử mọi hoán vị | `O(n! · n)` | `O(n!/∏c_i!)` | chỉ dùng làm trọng tài, `n <= 8` |
| 2a. Tham lam ngây thơ | chọn chữ được phép nhỏ nhất alphabet | `O(26 · n)` | `O(26)` | **SAI**: `"abb"`, `k = 2` |
| 2. Tham lam quét 26 | chọn chữ được phép còn nhiều nhất | `O(26 · n)` | `O(26)` | đúng; chậm khi bảng chữ lớn |
| **3. Heap + phòng chờ** | heap giữ chữ được phép, queue giữ chữ đang nghỉ | **`O(n log σ)`** | `O(σ)` | cách chính, tổng quát cho mọi bảng chữ |
| 4. Lối tắt `k <= 1` | mọi cách xếp đều hợp lệ | `O(1)` | `O(1)` | gắn vào đầu cách 3 |
| Bonus: công thức | `(maxf-1)·k + cnt_max <= n` | `O(n)` | `O(σ)` | chỉ trả lời **có/không**, không xây chuỗi |

## 15. Lỗi người mới hay gặp

Mỗi lỗi: **code sai → vì sao sai → sửa**. Cell code ngay sau dùng một hàm "có công tắc lỗi" để tái hiện từng lỗi trên input cụ thể.

**Lỗi 1 — Quên đổi dấu, dùng min-heap thẳng `(count, c)`.**
Heap khi đó lấy chữ còn **ít** nhất trước — đúng kiểu tham lam ngây thơ. `"aab"`, `k = 2`: lấy `b` trước, rồi `a`, rồi `a` bị khóa → `""`, trong khi `"aba"` hợp lệ. *Sửa:* lưu `(-count, c)` và nhớ đổi dấu khi đọc.

**Lỗi 2 — Chỉ đưa chữ vào phòng chờ khi còn lượt.**
```python
if neg + 1 < 0:            # SAI: bỏ qua chữ vừa hết lượt
    wait.append((neg + 1, c))
```
Đồng hồ chạy chậm: ngay Example 1 `"aabbcc"`, `k = 3` cũng hỏng — sau khi `a` hết lượt ở vị trí 3, phòng chờ chỉ còn 2 chữ nên `b` không được thả kịp → heap cạn → `""`. *Sửa:* **luôn** `append`, chỉ kiểm tra còn lượt lúc **thả** ra.

**Lỗi 3 — Lệch một: dùng `len(wait) > k`.**
Chữ phải chờ `k + 1` bước thay vì `k`. Example 1 lại hỏng: ba chữ `a, b, c` đều ngồi trong phòng chờ, heap rỗng ở vị trí 3 → `""`. *Sửa:* `>= k`.

**Lỗi 4 — Lệch một theo chiều ngược lại: `len(wait) >= k - 1`.**
Chữ được thả **sớm** một bước → output **vi phạm** khoảng cách. `"aa"`, `k = 2` trả `"aa"` (khoảng cách 1). Lỗi này nguy hiểm vì hàm vẫn trả chuỗi "trông có vẻ đúng" — chỉ hàm `is_valid_arrangement` mới bắt được.

**Lỗi 5 — Quên kiểm tra `len(result) == len(s)` ở cuối.**
`"aaabc"`, `k = 3` trả `"abca"` — chuỗi **thiếu chữ** thay vì `""`.

**Lỗi 6 — Dùng `len(wait) == k` và không có lối tắt `k <= 1`.**
Với `k = 0`: sau `append`, phòng chờ luôn có `>= 1` chữ nên `== 0` không bao giờ đúng → không ai được thả → `"aa"`, `k = 0` trả `""` (sai, vì `k = 0` thì mọi chuỗi đều hợp lệ). *Sửa:* dùng `>=` **và** thêm `if k <= 1: return s`.

**Lỗi 7 — Kiểm thử bằng so sánh với một chuỗi cố định.**
`assert out == "abcabc"` sẽ đánh trượt một lời giải đúng nhưng chọn chữ khác khi hòa (ví dụ ưu tiên chữ **lớn** hơn cho ra `"cbacba"`). Đề cho phép **bất kỳ** đáp án hợp lệ nào. *Sửa:* kiểm tra bằng `is_valid_arrangement` + trọng tài khả thi (brute force hoặc công thức).

**Lỗi 8 — Tham lam ngây thơ (mục 5):** chọn chữ được phép **bất kỳ** (theo alphabet, theo thứ tự xuất hiện…) thay vì chữ còn **nhiều nhất**. Phản ví dụ nhỏ nhất: `"abb"`, `k = 2`.

In [27]:
def heap_queue_variant(s, k, negate=True, always_wait=True, cmp=">=", shift=0, check_len=True, shortcut=True):
    """Cách 3 có các 'công tắc lỗi' để tái hiện lỗi thường gặp."""
    if shortcut and k <= 1:
        return s
    sign = -1 if negate else 1
    heap = [(sign * v, c) for c, v in Counter(s).items()]
    heapq.heapify(heap)
    wait, result = deque(), []
    while heap:
        key, c = heapq.heappop(heap)
        result.append(c)
        left = sign * key - 1                       # số lượt còn lại (số dương)
        if always_wait or left > 0:
            wait.append((left, c))
        L, K = len(wait), k + shift
        if wait and ((cmp == ">=" and L >= K) or (cmp == ">" and L > K) or (cmp == "==" and L == K)):
            left2, c2 = wait.popleft()
            if left2 > 0:
                heapq.heappush(heap, (sign * left2, c2))
    out = "".join(result)
    return (out if len(out) == len(s) else "") if check_len else out

def show(label, s, k, out):
    status = "rỗng" if out == "" else ("hợp lệ" if is_valid_arrangement(s, k, out) else "KHÔNG hợp lệ")
    print(f"{label:34} s={s!r:9} k={k}: {out!r:9} ({status}); đáp án đúng: {rearrange_heap_queue(s, k)!r}")

show("Lỗi 1: quên đổi dấu", "aab", 2, heap_queue_variant("aab", 2, negate=False))
show("Lỗi 2: bỏ chữ hết lượt khỏi hàng", "aabbcc", 3, heap_queue_variant("aabbcc", 3, always_wait=False))
show("Lỗi 3: len(wait) > k", "aabbcc", 3, heap_queue_variant("aabbcc", 3, cmp=">"))
show("Lỗi 4: len(wait) >= k - 1", "aa", 2, heap_queue_variant("aa", 2, shift=-1))
show("Lỗi 5: quên kiểm tra độ dài", "aaabc", 3, heap_queue_variant("aaabc", 3, check_len=False))

show("Lỗi 6: '==' và không lối tắt", "aa", 0, heap_queue_variant("aa", 0, cmp="==", shortcut=False))

# Lỗi 7: một lời giải ĐÚNG khác, hòa thì ưu tiên chữ LỚN hơn (đổi dấu cả chữ cái)
def rearrange_prefer_big(s, k):
    if k <= 1:
        return s
    heap = [(-v, -ord(c)) for c, v in Counter(s).items()]
    heapq.heapify(heap)
    wait, result = deque(), []
    while heap:
        neg, oc = heapq.heappop(heap)
        result.append(chr(-oc))
        wait.append((neg + 1, oc))
        if len(wait) >= k:
            n2, o2 = wait.popleft()
            if n2 < 0:
                heapq.heappush(heap, (n2, o2))
    return "".join(result) if len(result) == len(s) else ""

out = rearrange_prefer_big("aabbcc", 3)
print("Lỗi 7:", repr(out), "== 'abcabc'?", out == "abcabc", "| nhưng hợp lệ?", is_valid_arrangement("aabbcc", 3, out))

Lỗi 1: quên đổi dấu                s='aab'     k=2: ''        (rỗng); đáp án đúng: 'aba'
Lỗi 2: bỏ chữ hết lượt khỏi hàng   s='aabbcc'  k=3: ''        (rỗng); đáp án đúng: 'abcabc'
Lỗi 3: len(wait) > k               s='aabbcc'  k=3: ''        (rỗng); đáp án đúng: 'abcabc'
Lỗi 4: len(wait) >= k - 1          s='aa'      k=2: 'aa'      (KHÔNG hợp lệ); đáp án đúng: ''
Lỗi 5: quên kiểm tra độ dài        s='aaabc'   k=3: 'abca'    (KHÔNG hợp lệ); đáp án đúng: ''
Lỗi 6: '==' và không lối tắt       s='aa'      k=0: ''        (rỗng); đáp án đúng: 'aa'
Lỗi 7: 'cbacba' == 'abcabc'? False | nhưng hợp lệ? True


## 16. Mẹo nhận dạng pattern và các bài liên quan

**Dấu hiệu nhận ra pattern "tham lam + heap + phòng chờ":**

- Đề yêu cầu **sắp xếp lại / lên lịch** các phần tử có lặp lại, với ràng buộc kiểu "hai phần tử giống nhau phải cách nhau ít nhất `k`" hoặc "không được kề nhau".
- Có khái niệm **"hồi chiêu"** (cooldown): dùng xong phải chờ một khoảng cố định mới dùng lại được.
- Câu hỏi "có xếp được không" phụ thuộc vào **phần tử xuất hiện nhiều nhất** → nghĩ ngay tới công thức khung `(maxf - 1)·k + cnt_max`.

**Câu thần chú:** *"Lấy nhiều nhất — Cho vào phòng chờ — Đủ `k` thì thả — Hết heap mà chưa đủ thì bó tay."*

| Bài | Quan hệ với #358 |
|---|---|
| **767. Reorganize String** (Medium) | Đúng #358 với `k = 2` ("không có hai chữ kề nhau giống nhau"). Có thêm cách `O(n)`: xếp chữ nhiều nhất vào các chỉ số chẵn trước. (Bài tập 4) |
| **621. Task Scheduler** (Medium) | Cùng ràng buộc hồi chiêu nhưng được phép chèn **idle** (thời gian rảnh); hỏi **tổng thời gian nhỏ nhất**. Cooldown `n` của #621 tương ứng `k = n + 1` ở đây; đáp án là `max(len, (maxf-1)·(n+1) + cnt_max)`. (Bài tập 5) |
| **1054. Distant Barcodes** (Medium) | Giống #767 nhưng phần tử là **số nguyên** tới `10^4` → quét bảng chữ cố định (cách 2) không còn hợp, heap (cách 3) hoặc xếp chẵn/lẻ mới hợp. (Bài tập 6) |
| **984. String Without AAA or BBB** (Medium) | Tham lam "chữ còn nhiều hơn đi trước", ràng buộc dạng "không quá 2 liên tiếp". |
| **1405. Longest Happy String** (Medium) | Mở rộng #984 cho 3 chữ, cũng dùng max-heap chọn chữ nhiều nhất còn được phép. |

## 17. Bài tập vận dụng

Sáu bài, tăng dần độ khó. Hãy tự làm trước, bí thì xem **Gợi ý theo mức** (mục 18), cuối cùng mới mở **Đáp án** (mục 19). Các cell TODO chỉ định nghĩa hàm, **không** gọi hàm; test để sẵn dưới dạng comment — làm xong thì bỏ dấu `#` để tự kiểm tra.

### Bài 1 — Đoán nhanh: xếp được hay không? (không chạy code)

Với mỗi cặp dưới đây, hãy (a) tính `maxf`, `cnt_max` và vế trái công thức `(maxf - 1)·k + cnt_max`; (b) kết luận xếp được hay không; (c) nếu được, viết tay **một** đáp án.

| # | `s` | `k` |
|---:|---|---:|
| 1 | `"aaab"` | 2 |
| 2 | `"aabb"` | 3 |
| 3 | `"aabbc"` | 3 |
| 4 | `"aaabbbcc"` | 3 |
| 5 | `"aaa"` | 0 |
| 6 | `"abcd"` | 4 |

### Bài 2 — Tính tay cách 3

Với `s = "aaabbc"`, `k = 2`, chạy tay cách 3 và điền bảng (dùng ký hiệu `a2` = chữ `a` còn 2 lượt):

| i | heap trước | lấy | kết quả | phòng chờ (sau khi thêm) | bước 3 (thả ai?) |
|---:|---|---|---|---|---|
| 0 | ? | ? | ? | ? | ? |
| 1 | ? | ? | ? | ? | ? |
| … | | | | | |

Câu hỏi phụ: ở bước nào có hai chữ **hòa** số lượt trong heap? Ở bước nào một chữ **hết lượt** bị bỏ khỏi phòng chờ?

### Bài 3 — Điền TODO: heap + phòng chờ

Hoàn thiện cách 3 theo câu thần chú. Đừng nhìn lại code ở trên.

In [28]:
def exercise_rearrange_string(s: str, k: int) -> str:
    # TODO 1: nếu k <= 1 thì trả về s
    # TODO 2: tạo heap các tuple (-số_lượt, chữ) từ Counter(s), rồi heapify
    # TODO 3: wait = deque(), result = []
    # TODO 4: while heap: lấy đỉnh, nối chữ vào result, đưa (neg + 1, chữ) vào cuối wait
    # TODO 5: nếu len(wait) >= k: popleft; nếu còn lượt (neg2 < 0) thì push lại heap
    # TODO 6: trả "".join(result) nếu đủ len(s) chữ, ngược lại ""
    pass

# Tự làm xong rồi mới mở test:
# assert is_valid_arrangement("aabbcc", 3, exercise_rearrange_string("aabbcc", 3))
# assert exercise_rearrange_string("aaabc", 3) == ""
# assert is_valid_arrangement("aaadbbcc", 2, exercise_rearrange_string("aaadbbcc", 2))
# assert exercise_rearrange_string("aa", 0) == "aa"

### Bài 4 — Biến thể: LeetCode 767 — Reorganize String (Medium)

Given a string `s`, rearrange the characters of `s` so that any two adjacent characters are not the same. Return *any possible rearrangement of* `s` *or return* `""` *if not possible*.

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
```

```text
s = "aab"   -> "aba"
s = "aaab"  -> ""
```

Constraints: `1 <= s.length <= 500`, `s` consists of lowercase English letters.

Câu hỏi tư duy: "không có hai chữ kề nhau giống nhau" tương ứng với `k` bằng bao nhiêu trong #358? Công thức khả thi rút gọn thành điều kiện gì về `maxf` và `n`? Thử viết **không dùng heap**: đặt chữ nhiều nhất vào các chỉ số chẵn `0, 2, 4, …` trước, rồi lần lượt các chữ khác, hết chỉ số chẵn thì chuyển sang chỉ số lẻ `1, 3, 5, …`.

In [29]:
def exercise_reorganize_string(s: str) -> str:
    # TODO 1: đếm Counter; nếu maxf > (n + 1) // 2 thì trả ""
    # TODO 2: sắp các chữ theo số lượt giảm dần (chữ nhiều nhất đứng đầu)
    # TODO 3: điền lần lượt từng chữ vào res[idx], idx bắt đầu 0, mỗi lần idx += 2;
    #         nếu idx >= n thì chuyển idx = 1
    # TODO 4: trả "".join(res)
    pass

# Tự làm xong rồi mới mở test:
# assert is_valid_arrangement("aab", 2, exercise_reorganize_string("aab"))
# assert exercise_reorganize_string("aaab") == ""
# assert is_valid_arrangement("vvvlo", 2, exercise_reorganize_string("vvvlo"))

### Bài 5 — Biến thể: LeetCode 621 — Task Scheduler (Medium)

You are given an array of CPU `tasks`, each labeled with a letter from A to Z, and a number `n`. Each CPU interval can be idle or allow the completion of one task. Tasks can be completed in any order, but there's a constraint: there has to be a gap of **at least** `n` intervals between two tasks with the same label. Return the **minimum** number of CPU intervals required to complete all tasks.

```python
class Solution:
    def leastInterval(self, tasks: List[str], n: int) -> int:
```

```text
tasks = ["A","A","A","B","B","B"], n = 2   -> 8    (A B idle A B idle A B)
tasks = ["A","C","A","B","D","B"], n = 1   -> 6    (A B C D A B)
tasks = ["A","A","A","B","B","B"], n = 3   -> 10   (A B idle idle A B idle idle A B)
```

Constraints: `1 <= tasks.length <= 10^4`, `tasks[i]` is an uppercase English letter, `0 <= n <= 100`.

Câu hỏi tư duy: khác #358 ở đâu? (Ở đây được chèn **idle**, nên luôn làm được; câu hỏi chuyển thành "tối thiểu bao lâu".) "Gap ít nhất `n` intervals" giữa hai task giống nhau tương ứng với khoảng cách chỉ số `k = ?` trong #358. Hãy giải **hai cách** và so với nhau: (1) mô phỏng heap + phòng chờ theo **thời gian**, (2) công thức khung ở mục 11.

In [30]:
from typing import List

def exercise_least_interval_sim(tasks: List[str], n: int) -> int:
    # TODO 1: heap = [-số_lượt] cho mỗi task; wait = deque() chứa (-còn_lại, thời_điểm_được_chạy_lại)
    # TODO 2: time = 0; while heap or wait: time += 1
    # TODO 3:   nếu heap còn: lấy task nhiều nhất, giảm lượt; nếu còn lượt thì đưa vào wait với hạn time + n
    #           (nếu heap rỗng thì CPU idle ở thời điểm này)
    # TODO 4:   nếu wait[0] tới hạn (== time) thì đưa nó về heap
    # TODO 5: trả time
    pass

def exercise_least_interval_formula(tasks: List[str], n: int) -> int:
    # TODO: max(len(tasks), (maxf - 1) * (n + 1) + cnt_max)
    pass

# Tự làm xong rồi mới mở test:
# for f in (exercise_least_interval_sim, exercise_least_interval_formula):
#     assert f(["A","A","A","B","B","B"], 2) == 8
#     assert f(["A","C","A","B","D","B"], 1) == 6
#     assert f(["A","A","A","B","B","B"], 3) == 10

### Bài 6 — Mini challenge: LeetCode 1054 — Distant Barcodes (Medium)

In a warehouse, there is a row of barcodes, where the `i`-th barcode is `barcodes[i]`. Rearrange the barcodes so that no two adjacent barcodes are equal. You may return any answer, and it is guaranteed an answer exists.

```python
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
```

```text
barcodes = [1,1,1,2,2,2]        -> [2,1,2,1,2,1]   (hoặc bất kỳ cách hợp lệ nào)
barcodes = [1,1,1,1,2,2,3,3]    -> [1,3,1,3,1,2,1,2]
```

Constraints: `1 <= barcodes.length <= 10000`, `1 <= barcodes[i] <= 10000`.

Yêu cầu: viết lời giải **không phụ thuộc bảng chữ 26 ô** (phần tử là số nguyên tới `10^4`). Thử cả hai: (a) tổng quát hóa cách 3 cho list số nguyên với `k = 2`; (b) cách xếp chẵn/lẻ `O(n log n)` hoặc `O(n)`. Viết thêm một hàm kiểm tra hợp lệ cho list số và test đối chiếu trên input ngẫu nhiên **có đảm bảo có đáp án**.

In [31]:
def exercise_rearrange_barcodes(barcodes: List[int]) -> List[int]:
    # TODO 1: đếm Counter(barcodes); sắp các giá trị theo số lượt giảm dần
    # TODO 2: res = [0] * n; idx = 0
    # TODO 3: với mỗi giá trị (theo thứ tự trên), lặp đủ số lượt: res[idx] = giá trị; idx += 2;
    #         nếu idx >= n thì idx = 1
    # TODO 4: trả res
    pass

# Tự làm xong rồi mới mở test:
# out = exercise_rearrange_barcodes([1, 1, 1, 1, 2, 2, 3, 3])
# assert sorted(out) == [1, 1, 1, 1, 2, 2, 3, 3]
# assert all(out[i] != out[i + 1] for i in range(len(out) - 1))

## 18. Gợi ý theo mức

Mỗi bài có 3 mức: **(1)** hướng tư duy, **(2)** cấu trúc / cú pháp, **(3)** gần như code. Chỉ mở mức tiếp theo khi mức trước chưa đủ.

**Bài 1**
1. Đếm từng chữ trước. `maxf` là số lớn nhất; `cnt_max` là có **bao nhiêu chữ** cùng đạt số đó.
2. So vế trái với `n`. Nhớ `k = 0` làm vế trái chỉ còn `cnt_max`.
3. Ví dụ #3: `"aabbc"`, `maxf = 2`, `cnt_max = 2` (`a` và `b`) → `1·3 + 2 = 5 <= 5`: xếp được, vừa khít; thử khung `a b c / a b`.

**Bài 2**
1. Mỗi bước đúng 3 việc: lấy đỉnh → vào cuối phòng chờ → nếu phòng chờ có `>= 2` chữ thì thả đầu hàng.
2. Hòa số lượt thì chữ nhỏ hơn theo alphabet ra trước. Chữ hết lượt vẫn vào phòng chờ, nhưng khi ra thì bị bỏ.
3. Bước `i = 0`: heap `a3 b2 c1` → lấy `a`, phòng chờ `[a2]` (1 < 2, chưa thả). Bước `i = 1`: heap `b2 c1` → lấy `b`, phòng chờ `[a2, b1]` → thả `a2`.

**Bài 3**
1. Đọc lại câu thần chú ở mục 16 và bảng trace `"aabbcc"`.
2. `heapq.heapify(heap)`, `heapq.heappop(heap)`, `wait.append(...)`, `wait.popleft()`, `heapq.heappush(heap, ...)`.
3. Dòng then chốt: `wait.append((neg + 1, c))` — **không** có `if` bao quanh; điều kiện thả là `if len(wait) >= k:`.

**Bài 4**
1. Kề nhau là khoảng cách 1, nên "không kề nhau" là khoảng cách `>= 2` → `k = 2`. Bạn có thể gọi lại cách 3 với `k = 2` để kiểm tra.
2. Với `k = 2`: `(maxf - 1)·2 + cnt_max <= n`. Có thể chứng minh nó tương đương `maxf <= (n + 1) // 2` (số chỉ số chẵn).
3. Đặt chữ nhiều nhất trước vào `0, 2, 4, …` thì nó không bao giờ kề chính nó; các chữ sau điền tiếp phần chẵn còn lại rồi sang lẻ. Một chữ có thể "vắt" từ cuối dãy chẵn sang đầu dãy lẻ — vì nó **không** phải chữ nhiều nhất (hoặc chữ nhiều nhất đã lấp đủ chẵn), nó không đủ dài để chạm lại chính nó.

**Bài 5**
1. Gap ít nhất `n` interval giữa hai task giống nhau ⇔ chỉ số cách nhau `>= n + 1` → `k = n + 1`. Được chèn idle nên không bao giờ "bó tay".
2. Mô phỏng: mỗi đơn vị thời gian, nếu heap còn thì chạy task nhiều nhất; nếu heap rỗng mà phòng chờ còn → idle. Phòng chờ lưu **thời điểm** được chạy lại `time + n` thay vì đếm độ dài.
3. Công thức: `(maxf - 1)` khối, mỗi khối dài `n + 1`, cộng hàng cuối có `cnt_max` task; nhưng nếu task nhiều đến mức lấp hết idle thì đáp án là `len(tasks)` → lấy `max`.

**Bài 6**
1. Giống Bài 4 nhưng phần tử là số; `Counter` vẫn dùng được với số.
2. `sorted(cnt, key=lambda v: -cnt[v])` cho thứ tự giá trị theo số lượt giảm dần. Đề đảm bảo có đáp án nên không cần trả rỗng.
3. Kiểm tra hợp lệ: `sorted(out) == sorted(barcodes)` và `all(out[i] != out[i+1] ...)`. Sinh input ngẫu nhiên có đáp án bằng cách chỉ giữ những list có `maxf <= (n + 1) // 2`.

## 19. Đáp án tham khảo

Nên tự làm trước rồi mới xem phần này.

**Bài 1**

| # | `s` | `k` | `maxf` | `cnt_max` | `(maxf-1)·k + cnt_max` | `n` | Kết luận | Một đáp án |
|---:|---|---:|---:|---:|---:|---:|---|---|
| 1 | `"aaab"` | 2 | 3 | 1 | 5 | 4 | không | `""` |
| 2 | `"aabb"` | 3 | 2 | 2 | 5 | 4 | không | `""` |
| 3 | `"aabbc"` | 3 | 2 | 2 | 5 | 5 | được (vừa khít) | `"abcab"` |
| 4 | `"aaabbbcc"` | 3 | 3 | 2 | 8 | 8 | được (vừa khít) | `"abcabcab"` |
| 5 | `"aaa"` | 0 | 3 | 1 | 1 | 3 | được (`k = 0`) | `"aaa"` |
| 6 | `"abcd"` | 4 | 1 | 4 | 4 | 4 | được | `"abcd"` |

Ở #2, dù `k = 3 <= n`, hai chữ `a` cần chen 2 chữ khác — chỉ có `b, b` mà `b` lại cũng cần chen 2 chữ khác. Đúng như nhận xét ở mục 14: có chữ lặp mà `k > σ` (`3 > 2`) thì chắc chắn không được.

**Bài 2** — `s = "aaabbc"`, `k = 2`:

| i | heap trước | lấy | kết quả | phòng chờ (sau khi thêm) | bước 3 |
|---:|---|---|---|---|---|
| 0 | `a3 b2 c1` | `a` | `a` | `a2` | chưa thả (1 < 2) |
| 1 | `b2 c1` | `b` | `ab` | `a2, b1` | thả `a2` |
| 2 | `a2 c1` | `a` | `aba` | `b1, a1` | thả `b1` |
| 3 | `b1 c1` | `b` (hòa với `c`, `b` nhỏ hơn) | `abab` | `a1, b0` | thả `a1` |
| 4 | `a1 c1` | `a` (hòa với `c`) | `ababa` | `b0, a0` | bỏ `b` (hết lượt) |
| 5 | `c1` | `c` | `ababac` | `a0, c0` | bỏ `a` (hết lượt) |

Kết quả `"ababac"`. Hòa số lượt ở bước 3 và 4; chữ hết lượt bị bỏ ở bước 4 và 5. Cell dưới kiểm chứng cả hai bài bằng code đã có.

In [32]:
bai1 = [("aaab", 2), ("aabb", 3), ("aabbc", 3), ("aaabbbcc", 3), ("aaa", 0), ("abcd", 4)]
for s, k in bai1:
    cnt = Counter(s)
    maxf = max(cnt.values())
    cnt_max = sum(1 for v in cnt.values() if v == maxf)
    out = rearrange_heap_queue(s, k)
    print(f"{s!r:11} k={k}: vế trái={(maxf - 1) * k + cnt_max}, n={len(s)}, "
          f"công thức={can_rearrange_formula(s, k)!s:5}, brute={rearrange_bruteforce(s, k) != ''!s:5}, heap -> {out!r}")
print()
assert trace_heap_queue("aaabbc", 2) == "ababac"

'aaab'      k=2: vế trái=5, n=4, công thức=False, brute=False, heap -> ''
'aabb'      k=3: vế trái=5, n=4, công thức=False, brute=False, heap -> ''
'aabbc'     k=3: vế trái=5, n=5, công thức=True , brute=True , heap -> 'abcab'
'aaabbbcc'  k=3: vế trái=8, n=8, công thức=True , brute=True , heap -> 'abcabcab'
'aaa'       k=0: vế trái=1, n=3, công thức=True , brute=True , heap -> 'aaa'
'abcd'      k=4: vế trái=4, n=4, công thức=True , brute=True , heap -> 'abcd'

i=0: heap=[a3 b2 c1] lấy 'a' kq='a'       chờ=[a2] -> chưa thả (1<2)
i=1: heap=[b2 c1] lấy 'b' kq='ab'      chờ=[a2, b1] -> thả a2 về heap
i=2: heap=[a2 c1] lấy 'a' kq='aba'     chờ=[b1, a1] -> thả b1 về heap
i=3: heap=[b1 c1] lấy 'b' kq='abab'    chờ=[a1, b0] -> thả a1 về heap
i=4: heap=[a1 c1] lấy 'a' kq='ababa'   chờ=[b0, a0] -> bỏ b (hết lượt)
i=5: heap=[c1] lấy 'c' kq='ababac'  chờ=[a0, c0] -> bỏ a (hết lượt)
heap rỗng sau 6/6 ký tự -> trả 'ababac'


In [33]:
# Bài 3 — chính là cách 3; viết lại để đối chiếu, kèm kiểm thử ngẫu nhiên so với brute force.
def answer_rearrange_string(s: str, k: int) -> str:
    if k <= 1:
        return s
    heap = [(-v, c) for c, v in Counter(s).items()]
    heapq.heapify(heap)
    wait, result = deque(), []
    while heap:
        neg, c = heapq.heappop(heap)
        result.append(c)
        wait.append((neg + 1, c))
        if len(wait) >= k:
            neg2, c2 = wait.popleft()
            if neg2 < 0:
                heapq.heappush(heap, (neg2, c2))
    return "".join(result) if len(result) == len(s) else ""

assert is_valid_arrangement("aabbcc", 3, answer_rearrange_string("aabbcc", 3))
assert answer_rearrange_string("aaabc", 3) == ""
assert is_valid_arrangement("aaadbbcc", 2, answer_rearrange_string("aaadbbcc", 2))
assert answer_rearrange_string("aa", 0) == "aa"

random.seed(3)
for _ in range(400):
    n = random.randint(1, 7)
    s = "".join(random.choice("abc") for _ in range(n))
    k = random.randint(0, n)
    out = answer_rearrange_string(s, k)
    assert (out != "") == (rearrange_bruteforce(s, k) != "")
    assert out == "" or is_valid_arrangement(s, k, out)
print("Bài 3 PASS (ví dụ đề + 400 test ngẫu nhiên so với brute force)")

Bài 3 PASS (ví dụ đề + 400 test ngẫu nhiên so với brute force)


**Bài 4 — #767 Reorganize String.** `k = 2`. Điều kiện khả thi `(maxf - 1)·2 + cnt_max <= n` tương đương `maxf <= (n + 1) // 2`: nếu `maxf <= (n+1)//2` thì hoặc `cnt_max = 1` (vế trái `2·maxf - 1 <= n`), hoặc nhiều chữ cùng đạt `maxf` (khi đó `cnt_max · maxf <= n` nên vẫn đủ chỗ); chiều ngược lại hiển nhiên vì chữ nhiều nhất cần `maxf` chỉ số không kề nhau. Cách chẵn/lẻ: chữ nhiều nhất điền vào chỉ số chẵn trước nên không bao giờ kề chính nó; các chữ còn lại điền tiếp và có thể "vắt" sang dãy lẻ, nhưng một chữ vắt từ cuối dãy chẵn sang đầu dãy lẻ có số lượt `<=` chữ nhiều nhất, nên các vị trí của nó không chạm nhau.

```python
class Solution:
    def reorganizeString(self, s: str) -> str:
        n = len(s)
        cnt = Counter(s)
        if max(cnt.values()) > (n + 1) // 2:
            return ""
        res = [""] * n
        idx = 0
        for c in sorted(cnt, key=lambda c: -cnt[c]):
            for _ in range(cnt[c]):
                if idx >= n:
                    idx = 1
                res[idx] = c
                idx += 2
        return "".join(res)
```

In [34]:
def answer_reorganize_string(s: str) -> str:
    n = len(s)
    cnt = Counter(s)
    if max(cnt.values()) > (n + 1) // 2:
        return ""
    res = [""] * n
    idx = 0
    for c in sorted(cnt, key=lambda c: -cnt[c]):
        for _ in range(cnt[c]):
            if idx >= n:
                idx = 1
            res[idx] = c
            idx += 2
    return "".join(res)

assert is_valid_arrangement("aab", 2, answer_reorganize_string("aab"))
assert answer_reorganize_string("aaab") == ""
assert is_valid_arrangement("vvvlo", 2, answer_reorganize_string("vvvlo"))
random.seed(767)
for _ in range(1500):
    n = random.randint(1, 12)
    s = "".join(random.choice("abcd") for _ in range(n))
    a = answer_reorganize_string(s)
    b = rearrange_heap_queue(s, 2)               # #358 với k = 2
    assert (a == "") == (b == ""), (s, a, b)
    assert a == "" or is_valid_arrangement(s, 2, a)
print("Bài 4 PASS:", answer_reorganize_string("aab"), answer_reorganize_string("vvvlo"))

Bài 4 PASS: aba vlvov


**Bài 5 — #621 Task Scheduler.** `k = n + 1`. Mô phỏng theo thời gian: phòng chờ lưu `(-còn_lại, thời_điểm_được_chạy_lại)`; vì task vào phòng chờ theo thứ tự thời gian tăng dần nên đầu hàng luôn tới hạn sớm nhất. Khi heap rỗng mà phòng chờ còn, thời gian vẫn tăng — đó là các ô **idle**. Công thức `max(len(tasks), (maxf - 1)·(n + 1) + cnt_max)` là phiên bản "có idle" của khung ở mục 11.

In [35]:
def answer_least_interval_sim(tasks: List[str], n: int) -> int:
    heap = [-v for v in Counter(tasks).values()]
    heapq.heapify(heap)
    wait = deque()                       # (-còn_lại, thời điểm được chạy lại)
    time = 0
    while heap or wait:
        time += 1
        if heap:
            neg = heapq.heappop(heap) + 1
            if neg < 0:
                wait.append((neg, time + n))
        if wait and wait[0][1] == time:
            heapq.heappush(heap, wait.popleft()[0])
    return time

def answer_least_interval_formula(tasks: List[str], n: int) -> int:
    cnt = Counter(tasks)
    maxf = max(cnt.values())
    cnt_max = sum(1 for v in cnt.values() if v == maxf)
    return max(len(tasks), (maxf - 1) * (n + 1) + cnt_max)

for f in (answer_least_interval_sim, answer_least_interval_formula):
    assert f(["A","A","A","B","B","B"], 2) == 8
    assert f(["A","C","A","B","D","B"], 1) == 6
    assert f(["A","A","A","B","B","B"], 3) == 10
random.seed(621)
for _ in range(1000):
    tasks = [random.choice("ABCD") for _ in range(random.randint(1, 15))]
    n = random.randint(0, 5)
    assert answer_least_interval_sim(tasks, n) == answer_least_interval_formula(tasks, n), (tasks, n)
print("Bài 5 PASS (ví dụ đề + 1000 test: mô phỏng == công thức)")

Bài 5 PASS (ví dụ đề + 1000 test: mô phỏng == công thức)


**Bài 6 — #1054 Distant Barcodes.** Cách xếp chẵn/lẻ của Bài 4 dùng nguyên được cho số nguyên (đề bảo đảm có đáp án nên bỏ bước kiểm tra). Độ phức tạp `O(n + d log d)` với `d` = số giá trị khác nhau (do sắp xếp). Ta còn viết bản heap (tổng quát hóa cách 3 cho list số với `k = 2`) và đối chiếu cả hai trên input ngẫu nhiên có đáp án.

```python
class Solution:
    def rearrangeBarcodes(self, barcodes: List[int]) -> List[int]:
        n = len(barcodes)
        cnt = Counter(barcodes)
        res = [0] * n
        idx = 0
        for v in sorted(cnt, key=lambda v: -cnt[v]):
            for _ in range(cnt[v]):
                if idx >= n:
                    idx = 1
                res[idx] = v
                idx += 2
        return res
```

In [36]:
def answer_rearrange_barcodes(barcodes: List[int]) -> List[int]:
    n = len(barcodes)
    cnt = Counter(barcodes)
    res = [0] * n
    idx = 0
    for v in sorted(cnt, key=lambda v: -cnt[v]):
        for _ in range(cnt[v]):
            if idx >= n:
                idx = 1
            res[idx] = v
            idx += 2
    return res

def barcodes_heap(barcodes: List[int], k: int = 2) -> List[int]:
    heap = [(-c, v) for v, c in Counter(barcodes).items()]
    heapq.heapify(heap)
    wait, res = deque(), []
    while heap:
        neg, v = heapq.heappop(heap)
        res.append(v)
        wait.append((neg + 1, v))
        if len(wait) >= k:
            n2, v2 = wait.popleft()
            if n2 < 0:
                heapq.heappush(heap, (n2, v2))
    return res if len(res) == len(barcodes) else []

def valid_barcodes(orig, out):
    return sorted(out) == sorted(orig) and all(out[i] != out[i + 1] for i in range(len(out) - 1))

for ex in ([1, 1, 1, 2, 2, 2], [1, 1, 1, 1, 2, 2, 3, 3]):
    print(ex, "->", answer_rearrange_barcodes(ex), "| heap:", barcodes_heap(ex))
    assert valid_barcodes(ex, answer_rearrange_barcodes(ex)) and valid_barcodes(ex, barcodes_heap(ex))

random.seed(1054)
tested = 0
while tested < 1000:
    arr = [random.randint(1, 5) for _ in range(random.randint(1, 14))]
    if max(Counter(arr).values()) > (len(arr) + 1) // 2:
        continue                                  # đề bảo đảm có đáp án -> bỏ input không hợp
    assert valid_barcodes(arr, answer_rearrange_barcodes(arr)), arr
    assert valid_barcodes(arr, barcodes_heap(arr)), arr
    tested += 1
print("Bài 6 PASS (1000 input ngẫu nhiên có đáp án, cả hai cách)")

[1, 1, 1, 2, 2, 2] -> [1, 2, 1, 2, 1, 2] | heap: [1, 2, 1, 2, 1, 2]
[1, 1, 1, 1, 2, 2, 3, 3] -> [1, 2, 1, 2, 1, 3, 1, 3] | heap: [1, 2, 1, 3, 1, 2, 1, 3]
Bài 6 PASS (1000 input ngẫu nhiên có đáp án, cả hai cách)


## 20. Template ngắn nên nhớ

```python
import heapq
from collections import Counter, deque

def rearrange_k_apart(s, k):
    if k <= 1:                                   # mọi cách xếp đều hợp lệ
        return s
    heap = [(-v, c) for c, v in Counter(s).items()]
    heapq.heapify(heap)                          # max-heap nhờ đổi dấu
    wait, res = deque(), []                      # phòng chờ: chữ vừa dùng
    while heap:
        neg, c = heapq.heappop(heap)             # 1. Lấy chữ còn nhiều nhất
        res.append(c)
        wait.append((neg + 1, c))                # 2. Vào phòng chờ (KỂ CẢ khi hết lượt)
        if len(wait) >= k:                       # 3. Đủ k thì thả đầu hàng
            neg2, c2 = wait.popleft()
            if neg2 < 0:
                heapq.heappush(heap, (neg2, c2))
    return "".join(res) if len(res) == len(s) else ""   # 4. Thiếu chữ -> bó tay
```

**Ba ý phải nhớ:**

1. **Chữ nhiều nhất là nút cổ chai** → tham lam chọn chữ **còn nhiều nhất trong số đang được phép**; khả thi ⇔ `(maxf - 1)·k + cnt_max <= n`.
2. **Phòng chờ là đồng hồ**: mỗi vị trí thêm đúng 1 chữ, `len >= k` thì thả → chữ đặt ở `p` dùng lại sớm nhất ở `p + k`.
3. Time `O(n log σ)`, Space `O(σ)` phụ (heap + phòng chờ cộng lại không quá `σ` chữ).

## Checklist trước khi submit

- [ ] Có lối tắt `if k <= 1: return s`?
- [ ] Heap lưu `(-count, chữ)` (đã đổi dấu) để lấy chữ **nhiều nhất**?
- [ ] Chữ vừa đặt **luôn** được `append` vào phòng chờ, kể cả khi số lượt về 0?
- [ ] Điều kiện thả là `len(wait) >= k` (không phải `> k`, `>= k - 1` hay `== k`)?
- [ ] Khi thả, chỉ `heappush` lại chữ **còn lượt**?
- [ ] Cuối cùng kiểm tra `len(result) == len(s)`, nếu thiếu thì trả `""`?
- [ ] Test bằng **hàm kiểm tra hợp lệ** (đúng `Counter`, khoảng cách `>= k`), không so với một chuỗi cố định?
- [ ] Đã thử `k = 0`, `k = 1`, `k = n`, chuỗi 1 ký tự, chuỗi toàn một chữ, ca vừa khít công thức, ca `"abb"`, `k = 2`?
- [ ] Đúng signature `rearrangeString(self, s: str, k: int) -> str`, không còn `print` debug?